# LightGBM con futuro · diario de experimentos (Pablo y Juanma · `futuro/`)

> ⚠️ Usa información de **otros snapshots de la sesión** (pasado y futuro): rompe a propósito la regla 1 del equipo.
> Por eso vive en `futuro/` y guarda en `futuro/oof|preds|submissions`: el stacking normal de `main` no lo ve; solo
> `stacking/stack.py --futuro` lo combina (ver `futuro/README.md`).

**Qué es:** `futuro/m_lgb_future.py`, un LightGBM LambdaRank (parámetros de referencia de `EQUIPO.md`) que puntúa cada
tarea de un snapshot. Ninguna feature usa la etiqueta. Tiene **dos pasos**:

1. **Primer pase (LightGBM).** Bloques de features de `futuro/features_future.py` (opción `--feats`):

| Bloque | Archivo | Qué contiene |
|---|---|---|
| `base` | `features_future.py` | columnas originales del snapshot |
| `logic` | `features_future.py` | reglas dentro del snapshot: plazo por tipo, factibilidad, ventana típica, EDF… (`logic_*`) |
| `past` | `features_future.py` | cuánto lleva la tarea en la lista, cambios respecto al snapshot anterior, racha (`p_*`, `blk_*`, `gap_*`) |
| `future` | `features_future.py` | segundos hasta la siguiente instrucción al avión y su tipo, si encaja con la tarea, cuándo desaparece la tarea (`f_*`) |
| `clear` | `feat_clear.py` | segundos hasta que cambian nivel autorizado, rumbo, nivel de salida, zona, conflicto, ruta (`sx_*`) |
| `coord` | `feat_coord.py` | segundos hasta que cambia la coordinación de entrada/salida, assume, transfer (`sc_*`): **no aporta** |

2. **Segundo paso de sesión** (`--sesion`, `futuro/feat_session.py`, sección 9). Un logit condicional de 9 pesos sobre
   los scores del primer pase. Usa que **cada tarea se trabaja una sola vez en la sesión**: si ya fue la favorita en
   otra racha, aquí se baja. Encima se aplica el **suavizado temporal** `prob` dentro de la racha
   (`futuro/pruebas/suavizado.py`, secciones 7 y 9b).

**Vocabulario:** *snapshot* = un `sample_id` (una lista de tareas cada 3 s). *Racha* (`block`) = snapshots seguidos de
una sesión sin huecos de más de 4.5 s. *Tramo* = (racha, tarea correcta).

**Cómo usar este notebook**
1. Ejecuta *Preparación* una vez (calcula las features, ~1–2 min).
2. Cada bloque: **hipótesis** → **variantes** (UNA cosa cambia respecto a la base) → **tabla** → conclusión escrita.
3. Pruebas rápidas con 1 semilla: diferencias < ~0.005 son ruido. Lo que gane se confirma con 3 semillas.
4. Las pruebas largas se lanzan desde la terminal en gpu.dsic (varias a la vez, 8 hilos cada una) y aquí se apuntan
   sus cifras. Todo queda en `experiments/lgb_future_runs.csv` (`historial()` al final).

**Cómo leer las cifras:** `cv` = top-3 out-of-fold con `Data/folds.csv` (GroupKFold de 5 por controlador).
`min_atco` = peor controlador (importa porque el leaderboard privado es **un solo controlador**, el 023).
*CV anidada* = los parámetros de un posproceso se eligen con 4 folds y se miden en el 5.º (número honesto).

**Regla desde el 21/10 (sección 14): todo experimento se juzga con TODAS estas métricas, no solo con la media.**
`m_lgb_future.py` las imprime y las apunta en `experiments/lgb_future_runs.csv` (vía `futuro/metricas.py`), y
`futuro/pruebas/comparar.py <ref> <cand>` las da para cualquier par de OOF:

| Métrica | Columna del registro | Por qué |
|---|---|---|
| CV media y por fold | `cv`, `fold0`–`fold4` | la de siempre |
| peor controlador (y cuál) | `min_atco`, `worst_atco` | el privado es un solo controlador |
| mediana por sesión | `ses_median` | la media la mueven unas pocas sesiones (12d) |
| CV por escenario | `sc1`, `sc2`, `sc3` | el 023 vuela los tres; una mejora en uno solo no es robusta |
| frente a la referencia (`--ref`, por defecto `lgb_future_clear`: el 1.er pase con `<ref>`, el segundo paso con `<ref>_ses`) | `d_cv`, `d_ses_median`, `ses_up`/`ses_down`, `d_atco_mean`/`d_atco_median`, `atco_up`/`atco_down` | cuántas sesiones y controladores mejoran o empeoran |

Un cambio se adopta si mejora la media **y** la mediana por sesión, sube en más sesiones de las que baja y no hunde
al peor controlador. Ojo: las diferencias de ±0.2 entre versiones pueden ser ruido de semillas (14b).

**Qué no se repite de `LGB_adri.ipynb`** (solo se lee, no se toca): Adri ya probó, con datos limpios, sus grupos de
features (`snapshot`, `tipo`, `avion`, `tipo_estado`, `contexto`): sueltos aportan poco y todos juntos empeoran. Sus
candidatos de parámetros (`lento`, `regularizado`, `xendcg`) se probaron aquí con futuro (bloque 2).

> ## Punto de partida actual (martes 20/10, 23:20)
>
> **Mejor modelo: `lgb_future_clear10_ses` + suavizado = CV anidada 0.918, peor controlador 0.784.**
> **Kaggle público (controlador 020): 0.947** con el envío actual. Los mejores equipos tienen 0.957.
>
> ```
> features base+logic+past+future+clear ─► LightGBM LambdaRank, 10 semillas ─► 0.9086
>        ─► segundo paso de sesión (logit de 9 pesos, feat_session.py)    ─► 0.9158
>        ─► suavizado prob en la racha (decay 0.85, k 3)                  ─► 0.9181 (CV anidada)
> ```
>
> | Modelo | CV top-3 | peor controlador | Kaggle público | dónde |
> |---|---|---|---|---|
> | `lgb_future` (base + logic + past + future) | 0.901 | 0.769 | 0.942 | 0–3 |
> | `lgb_future_clear` (+ `clear`) | 0.907 | 0.769 | — | 6, 6c |
> | `lgb_future_clear` + segundo paso de sesión | 0.914 | 0.781 | — | 9 |
> | `lgb_future_clear10` (10 semillas) + segundo paso | 0.916 | 0.781 | — | 9b |
> | `lgb_future_clear10` + segundo paso + suavizado | 0.918 (anidada) | 0.784 | 0.947 | 9b |
> | `lgb_future_clear10` + oráculo + segundo paso + suavizado | 0.923 (anidada) | 0.769 | **0.940** | 12 |
> | **`lgb_future_clear10` + oráculo de coordinaciones + segundo paso + suavizado** | **0.925** (anidada) | **0.796** | 0.934 | **14d, 14e** |
>
> **Lo que sabemos (no repetir):**
> 1. **Lo decide la información, no el modelo.** Sin el bloque `future` el LightGBM cae a 0.661 (1). `past` y `logic`
>    apenas aportan. Ningún ajuste de parámetros (2–3), ni XGBoost (0.896), CatBoost (0.874), la red (0.887) o el
>    stacking (0.903) mejoran al LightGBM: con el futuro todos fallan en los mismos snapshots (5, 6c).
> 2. **`clear` suma +0.6 de forma robusta; `coord` no aporta** (6, 6b, 6c), ni solo ni con el segundo paso (9b).
> 3. **La estructura de las etiquetas es la mayor fuente nueva** (9): el 98 % de las tareas son la respuesta en un solo
>    episodio de la sesión, y tras un hueco entre rachas la respuesta cambia casi siempre. El segundo paso que lo usa da
>    +0.7 y sube al peor controlador. Ese segundo paso debe ser **pequeño**: un LightGBM en su lugar sobreajusta (0.906).
> 4. **El suavizado dentro de la racha** da +0.1 sobre `lgb_future_clear`, pero +0.23 encima del segundo paso (7, 9b).
> 5. **El gran hueco siguen siendo las coordinaciones** (entry/exit, ~la mitad de los fallos, aciertos 0.75 / 0.61).
>    Casi no dejan huella en los datos, y el 74 % de los fallos están en tramos donde el modelo no acierta en ningún
>    snapshot (9).
>
> **Archivos para el envío** (los sube Juanma): `futuro/submissions/sub_suav_lgb_future_clear10_ses.csv` (la mejor) y
> `futuro/submissions/sub_lgb_future_clear10_ses.csv` (sin suavizado, la segura).
>
> **Actualización 21/10, 01:15 · mejor modelo: oráculo solo en coordinaciones** (sección 14d): CV anidada **0.925**,
> peor controlador **0.796**, 25 sesiones suben / 9 bajan, mediana por sesión +0.99. El oráculo usa etiquetas de otros
> controladores del mismo escenario (regla 4, decisión del equipo). **Entregas finales recomendadas:**
> `sub_suav_lgb_future_clear10_orcoord_ses.csv` (la mejor) y `sub_suav_lgb_future_clear10_ses.csv` (la segura, sin oráculo).
> **Desde la sección 14, cada experimento se juzga con media, mediana por sesión, escenarios, peor controlador y
> sesiones que suben/bajan** (`futuro/metricas.py`, `futuro/pruebas/comparar.py`).
>
> **Mapa del notebook:** 0 punto de partida antiguo · 1 bloques de features · 2–4 parámetros · 5 errores y red ·
> 5b suavizado (primera versión) · 6–6c `coord` / `clear` y otros árboles · 7 suavizado con CV anidada · **9 segundo paso
> de sesión** · **9b confirmaciones, variantes y suavizado encima** · 10 siguientes ideas · 11 pruebas sin mejora · 12 parámetros y oráculo · 13 tanda f5 (Juanma) · **14 criterio robusto y oráculo de coordinaciones** · 15 otros ámbitos (Juanma) · 16 coordinaciones desde el modelo · historial.

## Preparación

In [ ]:
%load_ext autoreload
%autoreload 2
import sys, pandas as pd
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name in ("notebooks", "futuro") else Path.cwd()
sys.path.insert(0, str(ROOT / "src")); sys.path.insert(0, str(ROOT / "cv")); sys.path.insert(0, str(ROOT / "futuro"))
import m_lgb_future

data = m_lgb_future.build_data()      # una sola vez; todas las pruebas lo reutilizan
REF = {"LGB limpio (Adri)": 0.648, "red con futuro": 0.887}

In [ ]:
# Configuración base = parámetros de referencia de EQUIPO.md con los 4 bloques de features. Cada experimento cambia UNA clave.
BASE = dict(feats="base,logic,past,future", n_estimators=400, lr=0.05, leaves=31, min_child=50, subsample=0.8,
            colsample=0.7, trunc=5, reg_lambda=0.0, objective="lambdarank")


def probar(nombre, motivo, semillas=1, guardar=False, **cambios):
    """Lanza la CV con BASE + cambios; queda apuntado en experiments/lgb_future_runs.csv con su motivo.
    guardar=True genera futuro/oof/, futuro/preds/ y futuro/submissions/ (solo para la versión que va al stacking)."""
    cfg = {**BASE, **cambios}
    args = ["--name", nombre, "--tag", motivo] + (["--quick"] if not guardar else [])
    for k, v in cfg.items():
        if isinstance(v, bool):                       # flags sin valor (sesion=True -> --sesion)
            args += [f"--{k}"] if v else []
        else:
            args += [f"--{k.replace('_', '-')}", str(v)]
    a = m_lgb_future.parse(args)
    a.seeds = semillas
    return m_lgb_future.run(a, data)


def lanzar(bloque, variantes, semillas=1, sobre=None):
    """variantes: lista de (nombre, motivo, {cambios}). Siempre incluye la base para comparar en igualdad."""
    sobre = sobre or {}
    res = [("base", "referencia", probar(f"{bloque}/base", "referencia", semillas, **sobre))]
    for nombre, motivo, cambios in variantes:
        res.append((nombre, motivo, probar(f"{bloque}/{nombre}", motivo, semillas, **{**sobre, **cambios})))
    return tabla_bloque(res)


def tabla_bloque(res):
    t = pd.DataFrame(res, columns=["variante", "por qué", "cv"])
    ref = t.loc[t.variante == "base", "cv"].iloc[0]
    t["Δ vs base"] = t.cv - ref
    t["veredicto"] = t["Δ vs base"].map(lambda x: "—" if abs(x) < 1e-9 else ("mejora" if x > 0.005 else
                                                   ("empeora" if x < -0.005 else "ruido")))
    return (t.sort_values("cv", ascending=False).style
            .format({"cv": "{:.4f}", "Δ vs base": "{:+.4f}"})
            .background_gradient(subset=["Δ vs base"], cmap="RdYlGn", vmin=-0.02, vmax=0.02).hide(axis="index"))


def historial(filtro=""):
    """Todo lo probado (también desde la terminal), ordenado por CV."""
    r = pd.read_csv(m_lgb_future.LOG)
    r = r[r.name.str.contains(filtro, regex=False)] if filtro else r
    cols = ["when", "name", "tag", "cv", "min_atco", "worst_atco", "ses_median", "sc1", "sc2", "sc3", "d_cv", "d_ses_median",
            "ses_up", "ses_down", "atco_up", "atco_down", "seeds", "feats", "n_feats", "leaves", "min_child", "lr",
            "n_estimators", "colsample", "secs"]
    return r[[c for c in cols if c in r]].sort_values("cv", ascending=False).reset_index(drop=True)

## 0 · Punto de partida

Base con 3 semillas, generada desde la terminal (`python futuro/m_lgb_future.py`): es la versión `lgb_future` que ya está en
`futuro/oof/` y `futuro/preds/` para Juanma. La celda la mide con la métrica del equipo (`cv/metric.py`).

In [ ]:
from metric import report
report(pd.read_csv(ROOT / "futuro" / "oof" / "lgb_future.csv"), "lgb_future (base, 3 semillas)")

## 1 · ¿Qué bloque de información aporta?

**Hipótesis:** casi todo el salto viene del bloque `future` (la siguiente instrucción delata la tarea); `past` y `logic` aportan poco.
Quitar un bloque cada vez dice cuánto vale cada uno. Si quitar uno **no empeora**, sobra (menos ruido, más robustez).

| Variante | Qué se quita | Qué esperamos |
|---|---|---|
| sin `future` | f_* | caída grande (debería acercarse al LightGBM limpio + pasado) |
| sin `past` | p_*, blk_*, gap_* | caída pequeña |
| sin `logic` | logic_* | ruido: con futuro, las reglas a mano deberían sobrar |

In [ ]:
lanzar("feats", [
    ("sin_future", "¿cuánto aporta el futuro?",       dict(feats="base,logic,past")),
    ("sin_past",   "¿cuánto aporta el pasado?",        dict(feats="base,logic,future")),
    ("sin_logic",  "¿sobran las reglas a mano?",       dict(feats="base,past,future")),
])

**Conclusión bloque 1 · casi toda la información está en el futuro; pasado y reglas no aportan.**

| Variante (1 semilla) | CV | Δ vs base (0.900) | peor controlador | |
|---|---|---|---|---|
| sin `logic` | 0.897 | −0.002 | 0.740 | ruido |
| sin `past` | 0.897 | −0.003 | 0.757 | ruido |
| sin `future` | **0.661** | **−0.239** | 0.596 | empeora muchísimo |

- **El bloque `future` es prácticamente todo el modelo:** sin él cae a 0.661, casi al nivel del LightGBM limpio de Adri (0.648).
- **`past` y `logic` no aportan con el futuro presente** (−0.002/−0.003, ruido): lo que dicen ya lo cuenta mejor la siguiente
  instrucción del controlador. No compensa quitarlos tampoco (no mejora), así que la base se queda con los 4 bloques.
- Sin futuro, `base+logic+past` da 0.661 frente a 0.648 de Adri: ~1 punto, que viene sobre todo del pasado (también rompe la
  regla 1). Las `logic_*` son solo del snapshot: Adri ya las tiene en `features.py` de `main` por si quiere probarlas.
- El peor controlador varía bastante entre variantes (0.740–0.757) con 1 semilla: no sacar conclusiones de ahí.

## 2 · Parámetros (los candidatos de Adri, sobre datos con futuro)

**Hipótesis:** con features tan informativas el modelo se apoya en pocas columnas muy fuertes (`f_next_input`…); lo que
le obligue a repartir (menos columnas por árbol) o a ser más prudente puede generalizar mejor a controladores nuevos.

| Variante | Cambios (iguales que en `LGB_adri.ipynb`) | Por qué |
|---|---|---|
| `lento` | `lr=0.03`, `n_estimators=700` | aprender más despacio con más árboles: suele generalizar mejor |
| `regularizado` | `min_child=100`, `colsample=0.5`, `reg_lambda=5` | hojas más grandes, menos columnas por árbol, pesos más pequeños |
| `xendcg` | `objective="rank_xendcg"` | otra pérdida de ranking, más estable con etiquetas ruidosas |

~1 min por variante (1 semilla).

In [ ]:
lanzar("params", [
    ("lento",        "más lento con más árboles (Adri)",                 dict(lr=0.03, n_estimators=700)),
    ("regularizado", "hojas grandes, menos columnas, L2 (Adri)",         dict(min_child=100, colsample=0.5, reg_lambda=5.0)),
    ("xendcg",       "otra pérdida de ranking (Adri)",                   dict(objective="rank_xendcg")),
])

**Conclusión bloque 2 · la CV no se mueve, pero `regularizado` y `lento` mejoran al peor controlador.**

| Variante (1 semilla) | CV | Δ vs base (0.900) | peor controlador | Δ peor | |
|---|---|---|---|---|---|
| lento (`lr=0.03`, 700 árboles) | 0.900 | +0.001 | 0.763 | +0.018 | CV igual, más robusto |
| regularizado (`min_child=100`, `colsample=0.5`, `reg_lambda=5`) | 0.900 | 0.000 | **0.775** | **+0.030** | CV igual, más robusto |
| xendcg | 0.886 | **−0.013** | 0.740 | −0.006 | empeora |

- **En CV media ningún candidato gana:** con el futuro, el modelo base ya está bien ajustado.
- **El peor controlador sí mejora** con `regularizado` (+3 puntos) y `lento` (+1,8). Importa porque el leaderboard privado
  es **un solo controlador** (023) que no está en train: un modelo que no se hunda con ninguno es más seguro. Encaja con la
  hipótesis: obligarle a repartir entre columnas (`colsample=0.5`) evita que dependa solo de `f_next_input`.
- Ojo: con 1 semilla el peor controlador baila (en el bloque 1 iba de 0.740 a 0.757), así que hay que confirmarlo con 3 semillas.
- **`xendcg` se descarta.**

## 3 · Combinar lo que ha funcionado

Pon en `MEJOR` los cambios con veredicto **mejora** de los bloques 1 (p. ej. `feats=...` si sobra un bloque) y 2. Se comprueba con **3 semillas** frente a la base de 3 semillas
(sección 0), porque varios cambios que ayudan por separado no siempre suman.

In [ ]:
MEJOR = dict(
    # bloque 2: misma CV, mejor peor controlador -> se confirma con 3 semillas en la celda siguiente
    min_child=100, colsample=0.5, reg_lambda=5.0,     # regularizado
    lr=0.03, n_estimators=700,                        # lento
)

In [ ]:
# Comprobación con 3 semillas. Se puede saltar: el bloque 4 solo necesita la celda de MEJOR.
cv_comb = probar("comb/" + ",".join(f"{k}={v}" for k, v in MEJOR.items()), "combinación de ganadores", semillas=3, **MEJOR)
print(f"combinación: {cv_comb:.4f}   referencias: {REF}")

**Conclusión bloque 3 · la combinación no mejora: la mejora del peor controlador era ruido de 1 semilla.**

| (3 semillas) | CV | peor controlador | fold 0 | fold 1 | fold 2 | fold 3 | fold 4 |
|---|---|---|---|---|---|---|---|
| **`lgb_future`** (base) | **0.901** | **0.769** | 0.938 | 0.890 | 0.879 | 0.896 | 0.904 |
| `MEJOR` (regularizado + lento) | 0.901 | 0.766 | 0.938 | 0.894 | 0.884 | 0.885 | 0.904 |

- Misma CV y mismo peor controlador: con 3 semillas desaparece la ventaja que se veía con 1 (0.775 vs 0.746).
- Además tarda el doble (700 árboles). **Se queda `lgb_future` como versión final** y el bloque 4 no se lanza.
- Lectura general del barrido: con el futuro, los parámetros de referencia de `EQUIPO.md` ya están bien; el modelo depende de
  la información (bloque 1), no del ajuste.

## 4 · Versión para el stacking (`futuro/`)

Si la combinación mejora a la base, genera la versión definitiva con 3 semillas en el formato común. Nombre nuevo
(`lgb_future_v2`…) para no pisar `lgb_future`. Juanma la combina con `stack.py --futuro`.

In [ ]:
NOMBRE = "lgb_future_v2"
probar(NOMBRE, "versión stacking futuro: " + str(MEJOR), semillas=3, guardar=True, **MEJOR)

**Resultado:** no se lanza; `lgb_future` (CV 0.901, ya en `futuro/oof/` y `futuro/preds/`) es la versión para el stacking de esta rama.

## 5 · Análisis de errores y comparación con la red con futuro

**Pregunta:** ¿fallan el LightGBM y la red en los mismos snapshots? Si no, la combinación de Juanma ganará. Se mira el acierto
por tipo de tarea de cada modelo y en cuántos snapshots acierta uno solo.

In [ ]:
from metric import hits, labels

TIPOS = ["entry_conflict", "exit_conflict", "entry_coord", "exit_coord", "assume", "transfer", "conflict",
         "non_conformance", "qos", "entry_conditions", "exit_conditions", "aircraft_request", "zone", "return_to_route"]


def comparar(modelos=("lgb_future", "nn_future")):
    lab = labels().set_index("sample_id")
    tipos = pd.concat([pd.read_csv(ROOT / "Data" / f"{f}.csv", usecols=["sample_id", "target_index", "task_index"])
                       for f in ("train", "validation")]).query("target_index == 1").set_index("sample_id").task_index
    h = pd.DataFrame({m: hits(pd.read_csv(ROOT / "futuro" / "oof" / f"{m}.csv")) for m in modelos})
    a, b = modelos
    print(f"top-3:  {a} {h[a].mean():.3f}   {b} {h[b].mean():.3f}")
    print(f"aciertan los dos {(h[a] & h[b]).mean():.3f} | solo {a} {(h[a] & ~h[b].astype(bool)).mean():.3f} | "
          f"solo {b} {(~h[a].astype(bool) & h[b]).mean():.3f} | ninguno {((h[a] == 0) & (h[b] == 0)).mean():.3f}")
    h["tipo"] = tipos.reindex(h.index).map(dict(enumerate(TIPOS)))
    t = h.groupby("tipo").agg(muestras=(a, "size"), **{a: (a, "mean"), b: (b, "mean")})
    t["Δ (lgb − red)"] = t[a] - t[b]
    return t.sort_values("muestras", ascending=False).style.format(
        {a: "{:.2f}", b: "{:.2f}", "Δ (lgb − red)": "{:+.2f}"}).background_gradient(subset=["Δ (lgb − red)"], cmap="RdYlGn")


comparar(("lgb_future", "nn_future"))      # cambiar a lgb_future_v2 si se genera

**Conclusión del análisis · los dos modelos fallan casi en los mismos sitios; mezclarlos a partes iguales no suma.**

| | % de snapshots |
|---|---|
| aciertan los dos | 86,6 % |
| solo `lgb_future` | 3,5 % |
| solo `nn_future` | 2,2 % |
| fallan los dos | 7,7 % |

| Tipo correcto | muestras | `lgb_future` | `nn_future` | Δ |
|---|---|---|---|---|
| transfer | 3.527 | 0.98 | 0.98 | +0.00 |
| assume | 3.366 | 0.98 | 0.98 | +0.01 |
| zone_conflict | 2.121 | 0.92 | 0.90 | +0.02 |
| entry_coordination | 1.068 | 0.71 | 0.67 | +0.03 |
| return_to_route | 937 | 0.87 | 0.82 | +0.04 |
| exit_coordination | 745 | 0.57 | **0.59** | **−0.02** |
| exit_conditions | 448 | 0.80 | 0.76 | +0.04 |

- El LightGBM es igual o mejor en casi todos los tipos; la red solo gana en **exit_coordination**.
- Media de puntuaciones normalizadas (z-score dentro del snapshot), según el peso de la red: 0 → **0.901**, 0.3 → 0.900,
  0.5 → 0.897, 1 → 0.887. **Cuanto más peso a la red, peor:** con la información del futuro, los dos aprenden lo mismo.
- Para Juanma: con el futuro, `lgb_future` solo ya es lo mejor. Su meta-modelo (que puede fiarse de la red solo en
  exit_coordination) podría arañar algo, pero no hay que esperar mucho. Contraste con los modelos limpios, donde la red sí
  sumaba (LightGBM 0.648 → mezcla 0.655): sin futuro hay menos información y cada modelo captura cosas distintas.
- (Cifras calculadas con la misma función de la celda anterior; la salida de la celda no quedó guardada en el notebook.)

## 5b · Suavizado temporal de las predicciones

**Hipótesis:** la descripción de los datos dice que los snapshots consecutivos suelen compartir respuesta (el 72 % repite la del
anterior dentro de una racha contigua; rachas de 3 snapshots de mediana). Promediar la probabilidad de cada tarea con la de
la misma tarea en los snapshots vecinos (pesos que decaen con la distancia) debería corregir fallos aislados.

In [ ]:
import numpy as np, features_future as ff

_blk = ff.add_blocks(ff.add_folds(ff.load()))
_blk = _blk[_blk.split != "test"][["sample_id", "row_index", "task_name", "block", "blk_pos"]]


def suavizar(modelo, k=5, decay=0.7):
    """Media ponderada de la probabilidad de la misma tarea en los ±k snapshots contiguos."""
    x = _blk.merge(pd.read_csv(ROOT / "futuro" / "oof" / f"{modelo}.csv"), on=["sample_id", "row_index"])
    x["p"] = np.exp(x.score - x.groupby("sample_id").score.transform("max"))
    x["p"] /= x.groupby("sample_id").p.transform("sum")
    x = x.sort_values(["block", "task_name", "blk_pos"]); g = x.groupby(["block", "task_name"]).p
    num, den = x.p.copy(), pd.Series(1.0, index=x.index)
    for i in range(1, k + 1):
        for sh in (g.shift(i), g.shift(-i)):
            num += decay ** i * sh.fillna(0); den += decay ** i * sh.notna()
    return hits(x[["sample_id", "row_index"]].assign(score=x.p)).mean(), hits(x[["sample_id", "row_index"]].assign(score=num / den)).mean()


pd.DataFrame({m: suavizar(m) for m in ("lgb_future", "nn_future")}, index=["sin suavizar", "suavizado k=5"]).round(4)

**Conclusión 5b:** mejora real pero pequeña (calculado con esta función, k = 2 / 5 / 10):

| | sin suavizar | k=2 | k=5 | k=10 |
|---|---|---|---|---|
| `lgb_future` | 0.901 | 0.906 | 0.906 | **0.907** |
| `nn_future` | 0.887 | 0.895 | **0.898** | 0.898 |

- +0.6 puntos en el LightGBM, +1 en la red: las features de futuro ya contienen buena parte de esa información.
- Para usarlo en test haría falta aplicarlo también a `preds/` (mismas rachas en las sesiones de test). Candidato para la
  versión final de esta rama.

## 6 · Bloques nuevos: `coord` y `clear`

**Por qué:** en la sección 5 lo que queda son sobre todo las **coordinaciones** (entry + exit ≈ 45 % de los fallos) y
`f_next_input` no las ve: coordinar con un sector vecino no es una instrucción al avión. Los dos bloques nuevos miden,
para cada tarea, **cuántos segundos faltan hasta que cambia en la sesión el estado que esa tarea resuelve**:

| Bloque | Archivo | Columnas | Qué cambio futuro mide | Tipos a los que apunta |
|---|---|---|---|---|
| `coord` | `futuro/feat_coord.py` | `sc_*` | coordinación de entrada (`x_isc_*`), de salida (`k_out_*`), assume (`is_assumed` 0→1), transfer (1→0) | entry/exit_coordination, assume, transfer, gemelo assume/entry |
| `clear` | `futuro/feat_clear.py` | `sx_*` | nivel autorizado, rumbo, nivel de salida, fin de zona/conflicto, vuelta a ruta | exit_conditions, zone_conflict, return_to_route, conflictos |

**Hipótesis:** `coord` debería subir sobre todo entry/exit_coordination; `clear` los tipos de instrucciones. Se añaden de
uno en uno sobre la base (los 4 bloques de siempre), y después juntos. 1 semilla, ~1 min por variante.

In [ ]:
lanzar("nuevos", [
    ("+coord",       "cambios futuros de coordinación/assume/transfer",  dict(feats="base,logic,past,future,coord")),
    ("+clear",       "cambios futuros de instrucciones",                 dict(feats="base,logic,past,future,clear")),
    ("+coord+clear", "los dos bloques nuevos",                           dict(feats="base,logic,past,future,coord,clear")),
])

**Conclusión bloque 6 · `clear` mejora; `coord`, sorprendentemente, casi no.**

| Variante (1 semilla) | CV | Δ vs base (0.900) | peor controlador | |
|---|---|---|---|---|
| +clear | **0.906** | **+0.006** | 0.757 | mejora (justo por encima del umbral) |
| +coord+clear | 0.905 | +0.006 | 0.775 | mejora |
| +coord | 0.901 | +0.001 | 0.769 | ruido |

- **`clear` es la que aporta:** sube 4 de los 5 folds (fold 0 +0.012, fold 2 +0.023). Saber cuándo cambia el nivel, el rumbo o
  la ruta del avión delata las tareas de instrucciones (exit_conditions, zone, return_to_route).
- **`coord` sola no mueve la CV**, aunque iba a por las coordinaciones. Lo más probable es que esa información ya estuviera en
  `f_task_end` (la tarea de coordinación desaparece de la lista justo cuando se coordina).
- `+coord+clear` empata con `+clear` en CV y tiene mejor peor controlador (0.775), pero con 1 semilla ese número baila
  (ya pasó en el bloque 2). La comparación por tipo de 6b dirá si `coord` aporta algo en las coordinaciones.
- **6b se lanza con `coord+clear`**: si por tipo `coord` no mejora las coordinaciones, la versión final se queda solo con `clear`.

### 6b · Versión con los bloques que ganen y dónde mejora

Poner en `FEATS_NUEVAS` la combinación ganadora del bloque 6. Se genera con 3 semillas como **`lgb_future_coord`**
(no pisa `lgb_future`) y se compara por tipo de tarea con `lgb_future`: así se ve si mejora justo donde se esperaba
(coordinaciones / instrucciones). ~4 min.

In [ ]:
FEATS_NUEVAS = "base,logic,past,future,coord,clear"     # bloque 6: +clear y +coord+clear mejoran (+0.006)
probar("lgb_future_coord", "versión stacking futuro: " + FEATS_NUEVAS, semillas=3, guardar=True, feats=FEATS_NUEVAS)
comparar(("lgb_future_coord", "lgb_future"))


=== lgb_future_coord versión stacking futuro: base,logic,past,future,coord,clear  seeds=3 feats=base,logic,past,future,coord,clear n_estimators=400 lr=0.05 leaves=31 min_child=50 subsample=0.8 colsample=0.7 trunc=5 reg_lambda=0.0 objective=lambdarank folds= quick=False
  fold 0 (37s)
  fold 1 (73s)
  fold 2 (109s)
  fold 3 (145s)
  fold 4 (181s)
lgb_future_coord: CV top3 = 0.9045   (min atco 0.772)   f0:0.950  f1:0.892  f2:0.887  f3:0.894  f4:0.903   147 features
Guardado: lgb_future_coord (oof) (preds + sub)
top-3:  lgb_future_coord 0.905   lgb_future 0.901
aciertan los dos 0.888 | solo lgb_future_coord 0.017 | solo lgb_future 0.014 | ninguno 0.082


,muestras,lgb_future_coord,lgb_future,Δ (lgb − red)
tipo,,,,
transfer,3527,0.98,0.98,+0.00
assume,3366,0.98,0.98,+0.00
zone,2121,0.92,0.92,-0.00
entry_coord,1068,0.73,0.71,+0.03
return_to_route,937,0.87,0.87,-0.00
exit_coord,745,0.59,0.57,+0.02
exit_conditions,448,0.82,0.80,+0.02
conflict,390,0.86,0.85,+0.01
qos,257,0.84,0.83,+0.01


**Conclusión 6b · ganancia pequeña pero real, y justo donde se esperaba.**

| 3 semillas | CV | peor controlador | fold 0 | fold 1 | fold 2 | fold 3 | fold 4 |
|---|---|---|---|---|---|---|---|
| `lgb_future` | 0.901 | 0.769 | 0.938 | 0.890 | 0.879 | 0.896 | 0.904 |
| **`lgb_future_coord`** | **0.905** | 0.772 | 0.950 | 0.892 | 0.887 | 0.894 | 0.903 |

| Tipo | `lgb_future` | `lgb_future_coord` | Δ |
|---|---|---|---|
| entry_coordination | 0.706 | 0.732 | **+0.026** |
| exit_coordination | 0.569 | 0.592 | **+0.023** |
| exit_conditions | 0.799 | 0.821 | **+0.022** |
| zone_conflict | 0.920 | 0.916 | −0.004 |
| return_to_route | 0.868 | 0.866 | −0.002 |
| non_conformance (86 muestras) | 0.895 | 0.791 | −0.105 |

- **+0.35 puntos** con 3 semillas (la mitad de lo que sugería 1 semilla); mejoran 3 de 5 folds.
- Las coordinaciones suben más de 2 puntos frente a `lgb_future`, **pero eso no demuestra que sea por `coord`**: la
  comparación es contra la base, no contra `+clear` solo. La sección 6c (con `+clear` solo a 3 semillas) indica que
  `coord` no aporta e incluso resta un poco.
- `clear` mejora exit_conditions, pero no zone_conflict ni return_to_route, a las que también apuntaba.
- non_conformance cae 10 puntos, pero son 86 muestras (~9 aciertos): probablemente ruido; vigilarlo.
- ~~`lgb_future_coord` pasa a ser el mejor LightGBM con futuro~~ → lo supera `lgb_future_clear` (0.907, sección 6c).
  `lgb_future_coord` es el mismo modelo que en el estudio de gpu.dsic se llamó `lgb_future_v2` (mismas cifras).
- Pendiente (norma de probar en las dos redes): añadir `sc_*` y `sx_*` a la red con futuro.

## 6c · Estudio en gpu.dsic (20/10, 21:35–22:20): otros árboles y confirmación de `clear`

Lanzado desde la terminal en gpu.dsic con `futuro/m_gbm_future.py` y `futuro/m_lgb_future.py`, mismos folds y features.
Referencia: `lgb_future` = CV 0.901 (3 semillas), 0.8996 (1 semilla). **En Kaggle (público, controlador 020): 0.942.**
Sus salidas y registros se generaron en gpu.dsic y no están en el repo: aquí solo quedan las cifras.

**1 · ¿Otro modelo de árboles mejora al LightGBM? → No** (mismas 125 features, 3 semillas)

| Modelo | CV top-3 | Peor controlador | Folds |
|---|---|---|---|
| **lgb_future** (referencia) | **0.901** | **0.769** | 0.938 / 0.890 / 0.879 / 0.896 / 0.904 |
| XGBoost `rank:ndcg` (600 árboles, depth 6, GPU) | 0.896 | 0.731 | 0.929 / 0.881 / 0.891 / 0.887 / 0.896 |
| CatBoost `YetiRank` (1500 iteraciones, depth 6, GPU) | 0.874 | 0.689 | 0.905 / 0.868 / 0.848 / 0.875 / 0.876 |

| Stacking (`stack.py --futuro`) | Mejor stack | Peor controlador |
|---|---|---|
| lgb + xgb | 0.9010 (los pesos dan 0 a xgb) | 0.769 |
| lgb + xgb + cat | 0.9012 (`stack_avg` por rangos) | 0.754 |

Correlación de aciertos: lgb–xgb 0.86, lgb–cat 0.73. Ni solos ni combinados mejoran al LightGBM, y empeoran al peor
controlador. Igual que con la red con futuro (sección 5): **con la información del futuro todos los modelos aprenden lo
mismo; el límite está en la información, no en el modelo.** XGBoost y CatBoost se descartan.

**2 · `clear` confirmado con 3 semillas**

| Modelo | CV | Δ vs 0.901 | Peor controlador | Folds |
|---|---|---|---|---|
| **`lgb_future_clear`** (+clear) | **0.9072** | **+0.006** | 0.769 | 0.949 / 0.893 / 0.894 / 0.897 / 0.905 |
| `lgb_future_coord` (+coord +clear, sección 6b) | 0.9045 | +0.004 | 0.772 | 0.950 / 0.892 / 0.887 / 0.894 / 0.903 |

- **`clear` funciona:** +0.6 puntos, mejora o iguala en los 5 folds y no empeora al peor controlador.
- **`coord` no aporta, e incluso resta un poco** combinado con `clear`. Las coordinaciones (la mitad de los fallos que
  quedan) siguen sin verse. Hipótesis: el cambio de estado de coordinación ocurre lejos en el tiempo de cuando el
  controlador trabaja en ella, o cae en huecos sin snapshots etiquetados.
- **Modelo elegido: `lgb_future_clear` (CV 0.907)**, base del segundo nivel temporal (suavizado, sección 7)
  y del envío de las ~00:00.

**3 · Notas técnicas de gpu.dsic:** 2× Quadro RTX 5000 (16 GB), 32 núcleos. LightGBM va en CPU (8 hilos): caben 4 pruebas
a la vez. XGBoost 3.x de pip no trae GPU allí → `xgboost==2.0.3` con `task_index` en one-hot (su camino categórico falla
con numpy 2). CatBoost YetiRank en GPU calcula la métrica PFound en CPU: se calcula solo al final (`metric_period`).

## 7 · Suavizado temporal con test y CV anidada (Juanma)

Completa la 5b (mismo método `prob`, ahora con CV anidada y aplicado a test). Hay un snapshot cada ~3 s y el controlador suele seguir con la misma tarea varios seguidos, así que se mezcla
el score de cada tarea con el de la **misma tarea** (`session`, `task_name`) en los ±k snapshots de la misma racha (`block`):

- **prob** (el de la 5b, de Pablo): `(p(t) + Σ_j decay^j · p(t±j)) / (1 + Σ_j decay^j · [la tarea está en ese vecino])`, con `p` = softmax del score dentro del snapshot.
- **log** (alternativa): `lsm(t) + a · Σ_j decay^(j-1) · [lsm(t-j) + lsm(t+j)]`, con `lsm` = log-softmax.

Se comparan los dos con CV anidada y se usa el que gane para cada modelo.

Diferencias con la 5b:
- **CV anidada**: `a`, `decay` y `k` se eligen con 4 folds y se miden en el 5º. Así el número no está inflado por elegir entre muchas combinaciones.
- **Se aplica también a test** (`preds/`), con los vecinos de las sesiones de test. No usa etiquetas, así que no hay fuga. Los vecinos son de la misma sesión, por tanto del mismo controlador y fold, y su OOF tampoco vio la etiqueta.
- Las salidas van a `futuro/pruebas/oof|preds/suav_<modelo>.csv` y salen en `futuro/pruebas/panel_pruebas.ipynb`.

Código en `futuro/pruebas/suavizado.py`.

In [ ]:
import itertools
sys.path.insert(0, str(ROOT / "futuro" / "pruebas"))
import suavizado as S
from base_fut import datos, guardar, registrar

MODELOS = ["lgb_future", "nn_future"]          # cualquiera con oof (y preds) en futuro/ o futuro/pruebas/
GRIDS = {"prob": list(itertools.product((0.5, 0.7, 0.85, 1.0), (1, 2, 3, 5, 10))),                       # (decay, k)
         "log": list(itertools.product((0.1, 0.2, 0.3, 0.5, 0.75, 1.0), (1.0, 0.7, 0.5), (1, 2, 3, 5, 10)))}  # (a, decay, k)

_lab, _te, _ = datos()
_cols = ["sample_id", "row_index", "session", "block", "t", "task_name"]
LS = _lab[_cols + ["target_index", "fold"]].assign(y=_lab.target_index)
TS = _te[_cols].copy()
NB_L, NB_T = S.vecinos(LS, 10), S.vecinos(TS, 10)   # posición de la misma tarea en los ±10 snapshots vecinos

In [ ]:
def aplicar(metodo, score_lsm, nb, g):
    return S.suavizar_prob(np.exp(score_lsm), nb, *g) if metodo == "prob" else S.suavizar(score_lsm, nb, *g)


def suavizado_anidado(modelo):
    L = LS.merge(S.leer(modelo, "oof"), on=["sample_id", "row_index"], how="left")
    base = S.lsm(L, L.score.values)
    n, mejor = L.sample_id.nunique(), None
    for metodo, grid in GRIDS.items():
        sc = {g: aplicar(metodo, base, NB_L, g) for g in grid}
        filas, tot0, tot1 = [], 0, 0
        for f in range(5):
            tr, va = (L.fold != f).values, (L.fold == f).values
            g = max(grid, key=lambda g: S.hit3(L, sc[g], tr))
            h0, h1, m = S.hit3(L, base, va), S.hit3(L, sc[g], va), L[va].sample_id.nunique()
            tot0, tot1 = tot0 + h0 * m, tot1 + h1 * m
            filas.append(dict(fold=f, params=g, sin=h0, con=h1, mejora=h1 - h0))
        g_final = max(grid, key=lambda g: S.hit3(L, sc[g]))
        print(f"{modelo} · {metodo}: CV anidada {tot0 / n:.4f} -> {tot1 / n:.4f} ({(tot1 - tot0) / n:+.4f}); "
              f"parámetros finales (todo el OOF): {g_final}")
        display(pd.DataFrame(filas).round(4))
        if mejor is None or tot1 > mejor[0]:
            mejor = (tot1, metodo, g_final)
    print(f"-> {modelo}: se usa {mejor[1]} con {mejor[2]}\n")
    return L, base, mejor[1], mejor[2]


SUAV = {m: suavizado_anidado(m) for m in MODELOS}

In [ ]:
# aplica el método y los parámetros finales a OOF y test, guarda suav_<modelo> y lo registra para el panel
for modelo, (L, base, metodo, g) in SUAV.items():
    p = S.leer(modelo, "preds")
    T = TS.merge(p, on=["sample_id", "row_index"], how="left") if p is not None else None
    so = aplicar(metodo, base, NB_L, g)
    sp = aplicar(metodo, S.lsm(T, T.score.values), NB_T, g) if T is not None else None
    guardar(f"suav_{modelo}", L, so, T, sp)
    registrar(f"suav_{modelo}", hilo="suavizado", n_features=None,
              descripcion=f"{modelo} suavizado {metodo} {g} (notebook LGB_futuro)",
              extra={"metodo": metodo, "params": list(g)})
    print(f"suav_{modelo}: guardado", "con preds de test" if sp is not None else "SIN preds (no hay preds del modelo)")

**Conclusión 7** (CV anidada, primera medida):

| | sin suavizar | log | **prob (5b)** |
|---|---|---|---|
| `lgb_future` | 0.9010 | 0.9033 (+0.0023) | **0.9042 (+0.0032)** |
| `nn_future` | 0.8874 | 0.8776 (−0.0097) | **0.8973 (+0.0099)** |

- El método de la 5b (media de probabilidades normalizada) es el bueno. En la red el log empeora, porque sus scores tienen otra escala.
- Los 0.906–0.907 de la 5b salían de elegir k mirando todo el OOF. En CV anidada la mejora real es +0.3 puntos en el LightGBM y +1 en la red.
- Con preds de test ya se puede usar: siguiente paso, suavizar el stack de futuro.

## 9 · Segundo paso de sesión: «cada tarea se trabaja una sola vez» (20/10, 22:40–23:15 · gpu.dsic)

**Análisis de errores de `lgb_future_clear` (0.9072)** que lleva a la idea:

| Tipo correcto | muestras | acierto | % de los fallos |
|---|---|---|---|
| exit_coordination | 745 | 0.596 | 24.6 % |
| entry_coordination | 1068 | 0.735 | 23.1 % |
| zone_conflict | 2121 | 0.922 | 13.5 % |
| return_to_route | 937 | 0.877 | 9.4 % |
| exit_conditions | 448 | 0.812 | 6.9 % |

- **Las coordinaciones casi no dejan huella:** en exit_coordination, cuando no hay siguiente instrucción al avión
  (`f_next_act` vacío, 241 muestras) el acierto es 0.21, y `sc_task` está vacío en la mitad. Un barrido de *todas* las
  columnas del avión (¿cambia en ≤ 60 s?) solo encuentra señales que ya están en `f_*`/`sx_*` (nivel autorizado, etc.).
- **Los fallos son de tramo, no de snapshot:** el 74 % de los fallos están en tramos (racha, tarea) en los que el modelo
  no acierta en ningún snapshot. Por eso el suavizado dentro de la racha apenas suma.
- **Estructura de las etiquetas** (lo que no estábamos usando):
  - **el 98.2 % de las tareas son la respuesta en un solo episodio de la sesión**;
  - tras un hueco entre rachas la respuesta cambia casi siempre (solo el 0.2 % repite);
  - el 87 % de las rachas tienen una sola respuesta;
  - test tiene la misma estructura: 401 rachas, 4.6 snapshots por racha (train: 4.2).

**Hipótesis:** si el primer pase está muy seguro de que la tarea X fue la respuesta en *otra* racha de la sesión, aquí
probablemente no lo es (*explaining away*). No usa etiquetas: solo los scores OOF del primer pase en otras rachas de la
misma sesión, que están en el mismo fold.

**Variantes, todas sobre el OOF de `lgb_future_clear` (0.9072):**

| Variante | CV | peor controlador | |
|---|---|---|---|
| regla `lsm − λ·M` (M = máx. prob. media de la tarea en otra racha), λ elegido en CV anidada | 0.9135 | — | los 5 folds eligen λ=2 y mejoran |
| + pooling en la racha + 2–3 iteraciones (CV anidada) | 0.9135 | — | no añade |
| 2.º pase LightGBM (todas las features + 16 de sesión) | 0.9055 | 0.719 | **empeora**: se apoya en el score del 1.er pase y sobreajusta |
| **logit condicional con 9 features de sesión** (`futuro/feat_session.py`) | **0.9142** | **0.781** | pesos casi iguales en los 5 folds (M ≈ −2.6) |
| logit, 2 / 3 rondas | 0.9128 / 0.9114 | 0.781 | iterar empeora |

El mismo logit sobre otros primeros pases: `lgb_future` 0.901 → 0.911, `lgb_future_coord` 0.9045 → 0.912; la red
`nn_future` 0.887 → 0.878 (empeora, como con el suavizado `log`: sus scores tienen otra escala).

**Por tipo** (logit sobre `lgb_future_clear`): sube todo menos transfer (igual). entry_coordination +1.9,
exit_coordination +1.5, return_to_route +1.2, exit_conditions +1.8, qos +3.9, exit_conflict +5.1. Por controlador:
mejoran 15 de 18; la peor bajada es de −0.005 (atco 4).

Se usa desde la terminal con `--sesion`: hace el primer pase y el segundo, y guarda `<name>` y `<name>_ses`.
En CV, los pesos del logit se ajustan con los otros 4 folds; para test, con todo el OOF, y se aplican a `preds/`.

In [ ]:
# Estructura de las etiquetas que justifica el segundo paso (cifras de la tabla de arriba). Solo usa train+validation.
lab = data["lab"]
y = lab[lab.target_index == 1].sort_values(["session", "t"])
y["seg"] = ((y.task_name != y.groupby("session").task_name.shift()) | (y.blk_pos == 0)).cumsum()   # tramo = racha + tarea
seg = y.groupby("seg").agg(session=("session", "first"), task=("task_name", "first"), block=("block", "first"))
n_seg = seg.groupby(["session", "task"]).size()
print(f"tareas que son respuesta en un solo tramo de la sesión: {(n_seg == 1).mean():.3f}")
print(f"rachas con una sola respuesta: {(seg.groupby('block').size() == 1).mean():.3f}")
s0 = y[y.blk_pos == 0]
print(f"primer snapshot de una racha con la misma respuesta que la racha anterior: "
      f"{(s0.task_name == y.groupby('session').task_name.shift().loc[s0.index]).mean():.3f}")
te = data["te"].drop_duplicates("sample_id")
print(f"test: {len(te)} snapshots en {te.block.nunique()} rachas ({len(te) / te.block.nunique():.1f} por racha)")

In [ ]:
# ~3 min por fold con 3 semillas; guarda lgb_future_clear y lgb_future_clear_ses (oof, preds, submission)
probar("lgb_future_clear", "3s +clear", semillas=3, guardar=True, feats="base,logic,past,future,clear", sesion=True)

**Conclusión 9 · el segundo paso de sesión es la mayor mejora desde `clear`: +0.7 puntos, y sube al peor controlador.**

Ejecución limpia desde la terminal (gpu.dsic, 3 semillas, `--sesion`; registro en `experiments/lgb_future_runs.csv`):

| 3 semillas | CV | peor controlador | fold 0 | fold 1 | fold 2 | fold 3 | fold 4 |
|---|---|---|---|---|---|---|---|
| `lgb_future_clear` (1.er pase, regenerado) | 0.9071 | 0.769 | 0.949 | 0.893 | 0.894 | 0.897 | 0.905 |
| **`lgb_future_clear_ses`** | **0.9142** | **0.781** | 0.952 | 0.897 | 0.905 | 0.906 | 0.913 |

- Mejoran los 5 folds (+0.3 a +1.1) y el peor controlador (+1.2).
- Pesos ajustados con todo el OOF (los que se aplican a test): `lsm` 0.79, `lPb` 0.36, **`M` −2.67**, `E` 0.99,
  **`Pnext` −1.44**, `Pc` 0.26, `Mc` −0.23, `top_else` −0.08, `Pprev` 0.00. Lo que pesa es "la tarea ya fue la
  favorita en otra racha" (M) y "es la favorita en la racha siguiente" (Pnext): como tras un hueco la respuesta
  cambia, si en la racha siguiente gana esta tarea, aquí probablemente no es.
- Es complementario al suavizado (sección 7), que solo mira dentro de la racha; el pooling en la racha ya lo cuenta `lPb`.
- Salidas: `futuro/oof|preds/lgb_future_clear_ses.csv` y `futuro/submissions/sub_lgb_future_clear_ses.csv`
  (comprobada: 1863 snapshots, 3 tareas cada uno). `lgb_future_clear.csv` se ha regenerado (0.9071 frente a 0.9072:
  ruido de los hilos).
- ~~**Para el envío:** `lgb_future_clear_ses`~~ → superado por `lgb_future_clear10_ses` + suavizado (9b).

### 9b · Confirmaciones y variantes del segundo paso (gpu.dsic, 3 pruebas en paralelo × 8 hilos)

Lanzado desde la terminal con `--sesion`, p. ej.
`python futuro/m_lgb_future.py --feats base,logic,past,future,clear --sesion --seeds 10 --name lgb_future_clear10` (registro en `experiments/lgb_future_runs.csv`, logs en `logs/`). El 1.er pase y el
2.º se miden igual: misma CV con `Data/folds.csv`.

| Prueba | semillas | 1.er pase | + sesión | peor controlador (+ sesión) | |
|---|---|---|---|---|---|
| `clear` (referencia, `sesion/base`) | 1 | 0.9057 | 0.9132 | 0.778 | |
| `clear`, `colsample=0.5` (`sesion/colsample05`) | 1 | 0.9066 | 0.9125 | 0.775 | ruido: un 1.er pase "más repartido" no deja más margen al 2.º |
| `lgb_future_coord` (+coord +clear) | 3 | 0.9045 | 0.9123 | 0.772 | `coord` tampoco ayuda con sesión |
| `lgb_future_clear` | 3 | 0.9071 | 0.9142 | 0.781 | sección 9 |
| **`lgb_future_clear10`** | **10** | **0.9086** | **0.9158** | **0.781** | más semillas: +0.15 en los dos pasos |

**Features de sesión extra para las coordinaciones (prototipo sobre el OOF de `lgb_future_clear`, logit por fold):**

| Variante | CV | peor controlador | entry_coord | exit_coord |
|---|---|---|---|---|
| segundo paso de la sección 9 | 0.9142 | 0.781 | 0.754 | 0.611 |
| + gemelo assume↔entry_coordination y exit_coordination↔transfer (máx. prob. del gemelo en una racha anterior/posterior) | 0.9135 | 0.781 | 0.740 | 0.604 |
| + pesos propios para coordinaciones (`M`, `lsm`, `lPb` × [tipo 2 o 3]) | 0.9141 | 0.793 | 0.762 | 0.613 |
| + las dos | 0.9145 | 0.784 | 0.750 | 0.628 |

Todo dentro del ruido: no se añade (prototipo hecho fuera del repo; no se conserva su código porque se descarta). Lo que ya cuenta `Mc` (avión) recoge lo del gemelo.

**Suavizado temporal (sección 7, método `prob`) ENCIMA del segundo paso, en CV anidada:**

| Primer + segundo paso | sin suavizar | suavizado (CV anidada) | parámetros por fold | peor controlador |
|---|---|---|---|---|
| `lgb_future_clear_ses` (3 semillas) | 0.9142 | 0.9165 (+0.23) | (0.7,3), (0.85,5) ×4 | — |
| **`lgb_future_clear10_ses`** (10 semillas) | 0.9158 | **0.9181 (+0.23)** | (0.85,3) ×4, (1.0,3) | **0.784** |

- Ahora el suavizado suma algo más que sobre `lgb_future_clear` (+0.1). Tiene sentido: el 2.º paso mira *otras* rachas,
  y el suavizado, los snapshots vecinos dentro de la misma.
- Parámetros finales (elegidos con todo el OOF): decay=0.85, k=3, los mismos que en la sección 7. Con ellos la CV es 0.9183
  (optimista); el número honesto es el de la CV anidada, 0.918.
- Salidas: `futuro/oof|preds/lgb_future_clear10[_ses].csv`, `futuro/submissions/sub_lgb_future_clear10_ses.csv`;
  suavizado en `futuro/pruebas/oof|preds/suav_lgb_future_clear10_ses.csv` y
  `futuro/submissions/sub_suav_lgb_future_clear10_ses.csv` (comprobada: 1863 snapshots, 3 tareas cada uno).

**Candidatas para el envío (las elige Juanma):**
1. `sub_suav_lgb_future_clear10_ses.csv`: CV anidada **0.918**, peor controlador 0.784 (la mejor).
2. `sub_lgb_future_clear10_ses.csv`: CV 0.916, peor controlador 0.781 (sin suavizado: un paso menos).

In [ ]:
# Suavizado prob encima del segundo paso, con CV anidada; aplica los parámetros finales a test y guarda
# futuro/pruebas/oof|preds/suav_<modelo>.csv y futuro/submissions/sub_suav_<modelo>.csv (~2 min).
import itertools, numpy as np
sys.path.insert(0, str(ROOT / "futuro" / "pruebas"))
import suavizado as S
from base_fut import guardar

MODELO = "lgb_future_clear10_ses"
cols = ["sample_id", "row_index", "session", "block", "t", "task_name"]
L = data["lab"][cols + ["target_index", "fold"]].merge(pd.read_csv(ROOT / "futuro" / "oof" / f"{MODELO}.csv"), how="left")
L["y"] = L.target_index
T = data["te"][cols].merge(pd.read_csv(ROOT / "futuro" / "preds" / f"{MODELO}.csv"), how="left")
NB, NBT = S.vecinos(L, 10), S.vecinos(T, 10)
base = S.lsm(L, L.score.values)
grid = list(itertools.product((0.5, 0.7, 0.85, 1.0), (1, 2, 3, 5, 10)))       # (decay, k)
sc = {g: S.suavizar_prob(np.exp(base), NB, *g) for g in grid}
filas = []
for f in range(5):
    tr, va = (L.fold != f).values, (L.fold == f).values
    g = max(grid, key=lambda g: S.hit3(L, sc[g], tr))
    filas.append(dict(fold=f, params=g, n=L[va].sample_id.nunique(), sin=S.hit3(L, base, va), con=S.hit3(L, sc[g], va)))
t = pd.DataFrame(filas)
print(f"CV anidada {(t.sin * t.n).sum() / t.n.sum():.4f} -> {(t.con * t.n).sum() / t.n.sum():.4f}"); display(t.round(4))
g = max(grid, key=lambda g: S.hit3(L, sc[g]))
sp = S.suavizar_prob(np.exp(S.lsm(T, T.score.values)), NBT, *g)
guardar("suav_" + MODELO, L, sc[g], T, sp)
x = T[["sample_id", "row_index"]].assign(s=sp).sort_values(["sample_id", "s"], ascending=[True, False])
x.groupby("sample_id").row_index.apply(lambda r: " ".join(map(str, r.iloc[:3]))).rename("prediction").reset_index() \
    .to_csv(ROOT / "futuro" / "submissions" / f"sub_suav_{MODELO}.csv", index=False)
print("parámetros finales:", g)

## 10 · Siguientes ideas (20/10, 23:20)

Objetivo: pasar de 0.947 a ≥ 0.957 en Kaggle público. Se decide por la CV (y el peor controlador), no por el público.
Ya hecho: segundo paso de sesión (9), 10 semillas y suavizado encima (9b). Probado sin éxito (9b): features del gemelo,
pesos propios para las coordinaciones, `colsample=0.5`, `coord` con sesión. Pendientes que ya estaban: `clear` en la red
con futuro; las coordinaciones siguen sin señal propia.

Las ideas nuevas y sus resultados van en las secciones 11 y siguientes.


## 11 · Más allá del segundo paso (20/10, 23:30–00:00 · gpu.dsic)

**Referencia común:** `clear` con 1 semilla = 0.9057 en el 1.er pase y **0.9132 con el segundo paso** (`sesion/base`).
Sobre 10 semillas: 0.9086 → 0.9158. Pruebas rápidas desde la terminal (`--quick --sesion`) o prototipos sobre el OOF.

### 11a · Features de futuro por racha (`futuro/feat_block.py`, bloque `block`, columnas `bk_*`)

**Hipótesis:** los fallos son de tramo (sección 9), pero `f_*`/`sx_*`/`sc_*` se miden desde cada snapshot. Para cada
(racha, tarea) se toma el evento más temprano visto desde cualquier snapshot de la racha, medido respecto al **final de
la racha**: siguiente instrucción, 2.ª instrucción, desaparición de la tarea/avión, `sx_task`, `sc_task`; más el
tipo de esa instrucción y cuántas instrucciones recibe el avión durante la racha.

La señal existe: la tarea correcta desaparece justo al acabar la racha (mediana de `bk_f_task_end` 0 s, frente a 297 s
en el resto), y su siguiente instrucción, vista desde la racha, llega a 2 s del final (11 s vista desde el snapshot).

| 1 semilla | 1.er pase | + sesión | peor controlador (+ sesión) |
|---|---|---|---|
| referencia `clear` | 0.9057 | **0.9132** | 0.778 |
| `+block` | 0.9083 | 0.9116 | 0.772 |
| `+block+coord` | 0.9094 | 0.9129 | 0.769 |

**Conclusión:** el 1.er pase sube +0.3, pero tras el segundo paso no queda nada: lo que aportan ya lo recoge el
pooling por racha del segundo paso (`lPb`). **No se usa** (el bloque se queda en el código por si sirve a la red).

### 11b · Otras formas del segundo paso (prototipos sobre el OOF de `lgb_future_clear10`, 0.9086)

| Variante | CV | peor controlador | |
|---|---|---|---|
| **segundo paso actual** (logit de 9 pesos) | **0.9158** | 0.781 | |
| asignación global tipo Sinkhorn (cada racha suma 1, cada tarea ≤ 1 en la sesión), CV anidada | 0.9102 | 0.775 | peor: las restricciones duras no encajan |
| + `M` separada en rachas cercanas (≤ 60 s) y lejanas | 0.9159 | 0.781 | ruido |
| + probabilidad en la racha +2 | 0.9159 | 0.781 | ruido |
| + `Pnext` solo si el hueco hasta la siguiente racha es < 30 s | 0.9159 | 0.781 | ruido |
| + `lPb` × log(longitud de la racha) | 0.9158 | 0.784 | ruido |
| + todas | 0.9158 | 0.784 | ruido |

**Conclusión:** el segundo paso está saturado con lo que da la estructura de sesión.

### 11c · Pseudo-etiquetas del controlador nuevo

**Hipótesis:** el privado es un solo controlador que no está en train. Reentrenar el 1.er pase añadiendo los snapshots
del propio fold de validación, etiquetados con el top-1 de `lgb_future_clear10_ses` cuando su probabilidad > τ, adapta
el modelo a su estilo. Sin fuga: las pseudo-etiquetas del fold f salen de modelos que no vieron f.

| 1 semilla | pseudo-etiquetados | acierto de la pseudo-etiqueta | 1.er pase | + sesión | peor controlador |
|---|---|---|---|---|---|
| referencia | — | — | 0.9057 | 0.9132 | 0.778 |
| τ = 0.8 | 66 % | 0.963 | 0.9045 | 0.9138 | 0.787 |
| τ = 0.5 | 84 % | 0.898 | 0.9053 | 0.9129 | 0.787 |

**Conclusión:** ruido en la CV. El peor controlador sube un punto, pero con 1 semilla eso baila. No se usa.

### 11d · ¿Qué queda? Análisis de errores de `lgb_future_clear10_ses` (0.9158)

| Tipo correcto | muestras | acierto | % de los fallos |
|---|---|---|---|
| exit_coordination | 745 | 0.624 | 25 % |
| entry_coordination | 1068 | 0.746 | 24 % |
| zone_conflict | 2121 | 0.933 | 13 % |
| return_to_route | 937 | 0.892 | 9 % |
| exit_conditions | 448 | 0.835 | 7 % |

- En las entry_coordination falladas, la favorita suele ser **otra entry_coordination** (92 de 271): el modelo no sabe
  cuál de los aviones se coordina. En las exit_coordination falladas, el 64 % no tienen ninguna instrucción futura.
- **Solo el 10–13 % de las tareas de coordinación se trabajan alguna vez en la sesión** (assume: 90 %, transfer:
  96 %, zone_conflict: 68 %). No hay una restricción de "todas se hacen" que explotar.
- Que la tarea desaparezca antes que su avión ("se resuelve") **no predice** qué coordinación se trabaja
  (exit_coordination: ~10 % trabajadas en todos los tramos).
- **Conclusión:** las coordinaciones casi no dejan rastro en los datos; es el techo de la información disponible.

### 11e · Peso por tramo en el 1.er pase y combinación de primeros pases (20/10, 23:40)

**Peso por tramo** (`--peso racha|tramo --peso-pot p`): cada snapshot pesa 1/longitud^p, para que las rachas largas (muchos
snapshots casi iguales) no dominen el entrenamiento. `racha` = snapshots de la racha (sin etiqueta); `tramo` = snapshots
seguidos con la misma respuesta (usa la etiqueta, solo de las filas de entrenamiento). 1 semilla, 4 pruebas a la vez.

| 1 semilla | 1.er pase | + sesión | peor controlador (+ sesión) |
|---|---|---|---|
| referencia (sin peso) | 0.9057 | **0.9132** | 0.778 |
| `racha`, p = 0.5 | 0.9048 | 0.9097 | 0.778 |
| `racha`, p = 1 | 0.9069 | 0.9124 | 0.775 |
| `tramo`, p = 0.5 | 0.9064 | 0.9110 | 0.790 |
| `tramo`, p = 1 | 0.9055 | 0.9132 | 0.757 |

**Conclusión:** ruido o peor. No se usa (la opción queda en el código, apagada por defecto).

**Combinación de primeros pases antes del segundo paso** (z-score de cada modelo dentro del snapshot; referencia
`lgb_future_clear10` en z-score: 0.9086 → 0.9151 con sesión):

| + otro modelo | peso | 1.er pase | + sesión | peor controlador |
|---|---|---|---|---|
| `xgb_future` | 0.20 / 0.35 / 0.50 | 0.9085 / 0.9064 / 0.9035 | 0.9139 / 0.9117 / 0.9113 | 0.778 / 0.766 / 0.766 |
| `lgb_future_coord` | 0.20 / 0.35 | 0.9085 / 0.9073 | 0.9157 / 0.9143 | 0.790 / 0.790 |

**Conclusión:** lo mismo que en 6c (stacking): mezclar con otros modelos no suma, tampoco con el segundo paso encima. La
red y CatBoost no se lanzaron (ya eran peores en 5 y 6c).

## 12 · Parámetros con el segundo paso y el oráculo (20/10 23:45 – 21/10 00:30 · gpu.dsic)

### 12a · Parámetros del 1.er pase con el segundo paso encima

**Hipótesis:** los parámetros se eligieron sin el segundo paso. Ahora importa también el orden de las posiciones 2–10
(de ahí salen `M`, `Pnext`), y `trunc=5` solo optimiza las 5 primeras. 1 semilla, `--quick --sesion`, 3 a la vez.

| 1 semilla, + sesión | CV | peor controlador | |
|---|---|---|---|
| referencia (`leaves=31`, `min_child=50`, `trunc=5`, 400 árboles) | 0.9132 | 0.778 | |
| `leaves=15` | 0.9123 | 0.754 | ruido |
| `leaves=63` | 0.9151 | 0.760 | ruido |
| `min_child=20` | 0.9136 | 0.787 | ruido |
| `min_child=100` | 0.9104 | 0.754 | peor |
| `trunc=10` / `trunc=20` | 0.9135 / 0.9121 | 0.769 / 0.763 | ruido |
| 800 árboles, `lr=0.025` | 0.9138 | 0.787 | ruido |
| **`leaves=63`, `min_child=100`** | **0.9174** | 0.775 | mejora con 1 semilla → se confirma |

Confirmación con 3 semillas (`lgb_future_clear_l63`): 1.er pase 0.9076 (frente a 0.9071), **+ sesión 0.9159** (frente a
0.9142), peor controlador 0.766 (frente a 0.781). **No se confirma** (+0.17, ruido, y peor controlador más bajo): se
quedan los parámetros de siempre.

### 12b · El oráculo: qué hicieron otros controladores en el mismo escenario (`futuro/feat_oracle.py`, `--oraculo`)

**De dónde sale:** idea Pau #1 del EDA (`notebooks/01_eda.ipynb`, sección 9). En `main` se descartó **por la regla 4**
(usa `scenario` y `timestamp` para alinear), **no porque se midiera**: aquí se mide por primera vez.

**Por qué puede funcionar:** las sesiones de un escenario tienen el mismo tráfico. Los aviones de las 5 sesiones de
test están al 100 % en sesiones etiquetadas del mismo escenario (11–14 por escenario). El EDA ya veía que la mezcla de
tipos elegidos depende más del escenario (Cramér's V 0.159) que del controlador (0.096). Y ataca nuestro hueco: solo el
10–13 % de las coordinaciones se trabajan (11d), y si hay que hacer una de verdad, otros controladores también la hacen.

**Cómo se calcula:**
- **Tiempo alineado:** segundos desde el inicio de la sesión, corregidos con el desfase mediano de la primera
  aparición de los aviones comunes del escenario (sin etiquetas). Entre sesiones, un mismo avión aparece con ~22 s de
  diferencia.
- **Features `or_*`:** fracción de las otras sesiones en que esta tarea fue respuesta alguna vez (`or_ever`), a ±60 s
  y ±180 s del mismo momento (`or_near60/180`), lo mismo para cualquier tarea del avión (`or_cs_near60`), segundos
  hasta el momento mediano en que la trabajan los demás (`or_dt`, `or_dt_abs`) y rangos en el snapshot.
- **Sin fuga (usa etiquetas de OTRAS sesiones):** para el modelo del fold f solo cuentan sesiones etiquetadas del
  mismo escenario, de folds ≠ f y **de otro controlador que la fila** (también para las filas de entrenamiento).
  Para el modelo final y test: todas las sesiones etiquetadas salvo las del propio controlador.
- La integración en `m_lgb_future.py` reproduce exactamente el prototipo (`futuro/pruebas/oraculo.py`): 0.9070 /
  0.9192 con la misma semilla.

**Resultados:**

| | semillas | 1.er pase | + sesión | peor controlador (+ sesión) | folds (+ sesión) |
|---|---|---|---|---|---|
| sin oráculo | 1 | 0.9057 | 0.9132 | 0.778 | 0.955 / 0.893 / 0.903 / 0.903 / 0.916 |
| **con oráculo** | 1 | 0.9070 | **0.9192** | 0.784 | 0.954 / 0.905 / 0.899 / 0.916 / 0.925 |
| sin oráculo (`lgb_future_clear_ses`) | 3 | 0.9071 | 0.9142 | 0.781 | 0.952 / 0.897 / 0.905 / 0.906 / 0.913 |
| **con oráculo** | 3 | 0.9091 | **0.9236** | **0.763** ⚠️ | 0.954 / 0.905 / 0.908 / 0.923 / 0.931 |
| con oráculo + `leaves=63`, `min_child=100` | 3 | 0.9088 | 0.9211 | 0.760 | 0.958 / 0.902 / 0.903 / 0.925 / 0.923 |

- **+0.9 puntos con 3 semillas, y suben los 5 folds.** Casi todo aparece **en el segundo paso**: el oráculo apenas
  mueve el 1.er pase (+0.2), pero le da al segundo paso mejores candidatos en las posiciones 2–10.
- **Por tipo (1 semilla, + sesión):** exit_coordination 0.608 → **0.663 (+5.5)**, entry_coordination 0.741 → **0.767
  (+2.6)**, qos +3.1, exit_conflict +2.0, non_conformance +3.5. Bajan zone_conflict −0.8 y conflict_resolution −1.0.
  Mejora justo donde se esperaba: la mitad de los fallos eran coordinaciones.
- **Ganancia en el árbol:** `or_near180` 9 % del total, `or_dt` 3 %, `or_ever` 2 %; las demás, < 1 %.
- ⚠️ **El peor controlador baja con 3 semillas** (0.781 → 0.763). El privado es un solo controlador: se analiza abajo.

**Variantes del oráculo** (1 semilla, + sesión; base = 0.9192 / 0.784):

| Variante | Qué añade | CV | peor controlador |
|---|---|---|---|
| B | ventanas ±30/300/600 s, `or_cs_ever`, núcleo exp(−|Δt|/120) | 0.9187 | 0.763 |
| C | ventanas en tiempo relativo a la primera aparición del avión | 0.9175 | 0.737 |
| D | solo `or_near180`, `or_dt`, `or_ever` | 0.9188 | 0.749 |
| E | B + C | 0.9194 | 0.784 |

**Conclusión:** ninguna variante mejora a la base. **Se queda la base.**

**⚠️ Antes de subirlo (decisión del equipo):** el oráculo usa `scenario` y `timestamp` para alinear con otros
controladores (regla 4) y las etiquetas de otras sesiones. La CV es honesta (folds por controlador; el 023 vuela los
mismos escenarios), pero hay que poder defenderlo ante el jurado.

In [ ]:
# ~6 min con 3 semillas: guarda lgb_future_clear_or y lgb_future_clear_or_ses (oof, preds, submission)
probar("lgb_future_clear_or", "3s +clear +oráculo", semillas=3, guardar=True, feats="base,logic,past,future,clear",
       sesion=True, oraculo=True)

### 12c · Versión final con oráculo, análisis por controlador y comprobación en test

**Ejecuciones guardadas** (terminal, `--sesion --oraculo`; suavizado `prob` con CV anidada, mismo código que la celda de 9b):

| Modelo | semillas | 1.er pase | + sesión | + suavizado (CV anidada) | params suavizado (finales) | peor controlador |
|---|---|---|---|---|---|---|
| `lgb_future_clear10_ses` (envío actual, Kaggle 0.947) | 10 | 0.9086 | 0.9158 | 0.9181 | (0.85, 3) | 0.784 |
| `lgb_future_clear_or` | 3 | 0.9091 | 0.9236 | **0.9250** | (1.0, 10) | 0.769 |
| **`lgb_future_clear10_or`** | **10** | 0.9104 | 0.9215 | **0.9227** | (1.0, 5) | 0.769 |

- La diferencia 3 frente a 10 semillas (0.2) es ruido de las semillas. **Para enviar se elige la de 10 semillas**:
  sus predicciones de test son más estables. Mejora sobre el envío actual: **+0.5 en CV**.
- El suavizado suma ahora +0.12–0.14 (antes +0.23): parte de lo que aportaba ya lo da el oráculo.
- **Variante F** (oráculo dentro del logit de sesión: `or_near180` medio en la racha, `or_ever`, `or_near180`): 0.9227–0.9230
  frente a 0.9236. **No aporta**: lo que da el oráculo ya lo recoge el 1.er pase.

**Por controlador** (3 semillas + sesión, frente a `lgb_future_clear_ses`):

| atco | 001 | 016 | 008 | 010 | 004 | 002 | 005 | 017 | 007 | 015 | 006 | 003 | 018 | 020 | 019 | 011 | 009 | 014 |
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
| sin | 0.781 | 0.856 | 0.861 | 0.811 | 0.842 | 0.932 | 0.908 | 0.910 | 0.928 | 0.927 | 0.901 | 0.949 | 0.953 | 0.910 | 0.932 | 0.956 | 0.962 | 0.971 |
| con | 0.763 | 0.859 | 0.868 | 0.869 | 0.886 | 0.905 | 0.911 | 0.920 | 0.934 | 0.936 | 0.939 | 0.940 | 0.951 | 0.952 | 0.961 | 0.967 | 0.969 | 0.979 |
| Δ | −1.8 | +0.4 | +0.7 | **+5.7** | **+4.4** | −2.7 | +0.3 | +1.0 | +0.6 | +0.9 | **+3.7** | −1.0 | −0.3 | **+4.3** | **+2.9** | +1.0 | +0.7 | +0.8 |

- **Suben 14 de 18 controladores**, sobre todo los flojos. **El 020 (leaderboard público) sube +4.3.**
- Bajan 4 (001, 002, 003, 018). El peor sigue siendo el **001** (una sola sesión, 338 snapshots): pierde 6 snapshots
  (4 entry_coordination, 2 zone_conflict). Por sesiones, bajan 14 de 37 (la que más, 002sc3: −4.1).
- **Lectura:** el oráculo empuja hacia lo que hace "el controlador típico" del escenario: ayuda mucho a la mayoría y
  quita algo a los atípicos. **Riesgo para el privado:** que el 023 sea atípico. Ganancia media +0.9 frente a una
  pérdida máxima de −2.7.

**Comprobación en test (antes de subir):** las features del oráculo en test tienen la misma distribución que en el OOF
(`or_near180`: media 0.154 frente a 0.166 y 41 % frente a 40 % > 0; `or_ever`: 0.385 frente a 0.398; vacíos 0.8 % frente
a 0.5 %), y el tiempo alineado de las 5 sesiones de test cae en el rango de train (0–3460 s). No hay indicios de fallo.

**Envíos:**
- `futuro/submissions/sub_suav_lgb_future_clear10_or_ses.csv`: **la mejor** (CV anidada 0.923).
- `futuro/submissions/sub_suav_lgb_future_clear10_ses.csv`: **la segura**, sin oráculo (0.918, Kaggle 0.947).

En test, el 46 % de los snapshots cambian su top-3 respecto al envío actual. **Cómo leer el público** (±2–3 de ruido):
≥ 0.94 → el oráculo funciona en test; muy por debajo de 0.947 (p. ej. < 0.92) → algo falla y no se usa.

### 12d · Kaggle y ganancia por escenario (21/10, 00:50)

> ⚠️ **Corregido en 14c:** sin suavizar, el oráculo mejora en los tres escenarios. Lo de "solo el escenario 2" venía
> del suavizado y del ruido de las semillas. La recomendación de entregar una versión con oráculo y otra sin él se mantiene.

**Kaggle público (controlador 020, sesiones 020sc2 y 020sc3):** `sub_suav_lgb_future_clear10_or_ses.csv` = **0.940**,
frente a 0.947 del envío sin oráculo. Son −0.7 (~6 snapshots), dentro del ruido del público (±2–3). Pero la CV del 020
esperaba +4.3, así que es una señal (≈ 2 desviaciones) de que el oráculo transfiere peor de lo que decía la CV.

**Ganancia por escenario** (OOF, ambos con 10 semillas + sesión + suavizado):

| Escenario | sesiones | sin oráculo | con oráculo | Δ media | sesiones que suben / bajan | Δ mediana por sesión |
|---|---|---|---|---|---|---|
| 1 | 14 | 0.9033 | 0.9118 | +0.9 | 6 / 6 | **0.0** |
| 2 | 12 | 0.9233 | 0.9325 | +0.9 | 8 / 4 | +1.1 |
| 3 | 11 | 0.9294 | 0.9289 | **−0.05** | 3 / 6 | **−0.5** |

- **Solo el escenario 2 mejora de forma consistente.** En el 1, la media sube por unas pocas sesiones con ganancias
  grandes (controladores 010, 004…), pero la sesión típica no cambia. En el 3 no aporta o resta.
- Encaja con el público: 020sc2 (escenario 2) + 020sc3 (escenario 3). El +4.3 del 020 en la CV venía de su sesión
  020sc1, uno de los casos que ganan mucho: era engañoso.
- **Para el privado (023: escenarios 1, 2 y 3):** ganancia esperada pequeña, positiva e incierta, con riesgo a la baja.
- En test cambia el top-3 del 40–51 % de los snapshots de cada sesión, y el top-1 del 5–12 %.

**Decisión recomendada para las 2 entregas finales:** en Kaggle la nota final es la **mejor de las 2 seleccionadas**,
así que se elige **una sin oráculo y otra con oráculo** (cobertura gratis):
1. `sub_suav_lgb_future_clear10_ses.csv`: sin oráculo, CV 0.918, público 0.947.
2. `sub_suav_lgb_future_clear10_or_ses.csv`: con oráculo, CV 0.923, público 0.940.

**Lección:** con folds por controlador, la media de la CV la mueven unas pocas sesiones. Para decidir hay que mirar
también la **mediana por sesión** y **por escenario**, no solo el peor controlador.

## 13 · Tanda f5 de Juanma: `extra_trees`, árboles grandes, red con `clear` y stacks con sesión (20/10 23:00 – 21/10 00:30)

Desde la terminal con `futuro/pruebas/f5_clear_et.py` (variantes del 1.er pase con `base_fut.lgb_cv`: mismos parámetros,
features y folds que `m_lgb_future.py`) y `futuro/pruebas/f5_stack.py` (modos `ses` = segundo paso de `feat_session.sesion`
sobre cualquier oof/preds, `stack` = `stacking/stack.py` y `suav` = suavizado `prob` con CV anidada). Salidas en
`futuro/pruebas/oof|preds|submissions` y registro en `futuro/pruebas/resultados.jsonl` (hilos `f5_clear_et`, `f5_stack`).
Todo `+ clear` (base + logic + past + future + clear). Control: `f5_control_clear` reproduce `lgb_future_clear` (0.9071).

**13a · 1.er pase y segundo paso de sesión**

| Modelo | semillas | 1.er pase | + sesión | + suavizado (CV anidada) | peor controlador |
|---|---|---|---|---|---|
| `lgb_future_clear10` (referencia, 9b) | 10 | 0.9086 | 0.9158 | 0.9181 | 0.784 |
| `f5_clear_et` (`extra_trees`) | 3 | 0.9081 | 0.9161 | — | 0.787 |
| `f5_clear_et10` | 10 | 0.9077 | 0.9147 | 0.9157 | 0.778 / 0.790 |
| `f5_clear_l63_noet` (63 hojas, `min_child=100`) | 3 | 0.9077 | 0.9159 | — | 0.766 |
| `f5_clear_l127` (127 hojas + `extra_trees`) | 3 | 0.9076 | 0.9160 | — | 0.781 |
| `f5_clear_l63` (63 hojas, `min_child=100`, `extra_trees`) | 3 | 0.9106 | 0.9196 | 0.9207 | 0.778 |
| **`f5_clear_l63_10`** | **10** | **0.9113** | **0.9184** | **0.9202** | 0.778 / 0.769 |
| `f5_clear_binary` / `f5_clear_xendcg` | 3 | 0.9044 / 0.8978 | 0.8713 / 0.8934 | — | 0.680 / 0.710 |

- Solo gana la **combinación** 63 hojas + `extra_trees`; cada cosa por separado se queda en ~0.916 con sesión (igual que 12a).
- `binary` y `xendcg` empeoran mucho con el segundo paso (escala de scores distinta a la de LambdaRank).

**13b · Red con futuro + `clear` (lo pendiente de la sección 10)**: `m_nn_future.py --clear` (opción nueva, añade `sx_*`).
`nn_future_clear` / `_d128` / `_bs256` / `_h8` + sesión: 0.872–0.882 (la red sin `clear` + sesión: 0.878; sin sesión 0.887).
**El segundo paso empeora a la red** y en todos los stacks recibe peso ~0. Descartada.

**13c · Stacks** (media ponderada de `stack.py`, CV anidada)

| Stack | CV | + suavizado (anidada) | peor controlador |
|---|---|---|---|
| clear, f1_et, nn, linear, f4 (sin sesión) | 0.9085 | 0.9104 | 0.772 |
| sesión a cada modelo → stack (clear10, et10, redes, linear, f4) | 0.9186 | 0.9169 | 0.784 |
| stack → sesión (los mismos sin sesión) | 0.9164 | 0.9189 | 0.784 |
| 11 modelos con sesión (+ binary, xendcg, l63, redes) | 0.9179 | 0.9197 | 0.760 |
| `l63_10_ses` + `clear10_ses` + `linear_ses` + `f4_ses` | 0.9195 | 0.9197 | 0.772 |

Ningún stack mejora a `suav_f5_clear_l63_10_ses` solo; los dos órdenes (sesión antes o después del stack) quedan iguales.

**13d · Por escenario y sesión (criterio de 12d)**, `suav_f5_clear_l63_10_ses` frente a `suav_lgb_future_clear10_ses`:

| Escenario | Δ media | Δ mediana por sesión | sesiones que suben / bajan |
|---|---|---|---|
| 1 | −0.24 | −0.37 | 4 / 8 |
| 2 | **+1.26** | **+1.26** | 8 / 3 |
| 3 | −0.12 | −0.51 | 4 / 6 |

Total: 16 sesiones suben y 17 bajan, mediana 0. Controlador 020 (público): +3.8 en CV.
**Mismo patrón que el oráculo (12d):** el +0.25 de la media sale del escenario 2; en 1 y 3 resta un poco. No es robusto.


In [ ]:
# Tabla de la tanda f5 desde el registro (no reentrena nada)
import json
_r = {}
for l in open(ROOT / "futuro" / "pruebas" / "resultados.jsonl"):
    x = json.loads(l)
    if x["hilo"] in ("f5_clear_et", "f5_stack"):
        _r[x["nombre"]] = x
pd.DataFrame([dict(nombre=n, cv=x["cv"], cv_anidada=x.get("cv_anidada"), peor_controlador=x["min_atco"],
                   std_folds=x["std_folds"], preds=x["tiene_preds"]) for n, x in _r.items()]) \
    .sort_values("cv", ascending=False).reset_index(drop=True)


In [ ]:
# Reproducir (terminal, desde la raíz; ~15 min con 28 hilos):
#   python futuro/pruebas/f5_clear_et.py f5_clear_l63_10
#   python futuro/pruebas/f5_stack.py ses f5_clear_l63_10
#   python futuro/pruebas/f5_stack.py suav f5_clear_l63_10_ses
# Envío: futuro/pruebas/submissions/sub_suav_f5_clear_l63_10_ses.csv


## 14 · Lectura de la tanda f5 de Juanma, criterio por controlador y oráculo de coordinaciones (21/10, 00:55–01:30)

**14a · Qué deja la sección 13 (Juanma)**
- `f5_clear_l63_10` (63 hojas + `min_child=100` + `extra_trees`, 10 semillas) + sesión + suavizado = **0.9202** (CV
  anidada), frente a 0.9181. Por separado, los cambios no ganan (lo mismo que en 12a).
- La red con `clear` y los stacks no mejoran al LightGBM solo (igual que en 5 y 6c).
- Mismo patrón por escenario que el oráculo en 12d. **Además, los dos ganan y pierden en los mismos controladores**
  (ganan 006, 004, 020, 010; pierden 002, 001, 003, 018): son el mismo tipo de cambio, más seguros en los casos típicos.

**14b · Nuevo criterio: ganancia por controlador** (`futuro/pruebas/comparar.py`, reutilizable por todo el equipo)

El privado es un controlador nuevo. `comparar.py <ref> <cand>` da la CV, la ganancia por controlador (media = esperanza
para un controlador nuevo, mediana, suben/bajan), un bootstrap sobre los 18 controladores (IC 90 %, P(media > 0)) y la
ganancia por escenario.

**Limitación importante:** el conjunto de semillas mueve a **todos** los controladores a la vez, y el bootstrap por
controladores no lo ve. Ejemplo: `lgb_future_clear10_or_ses` frente a `lgb_future_clear_or_ses` (mismo modelo, 10
frente a 3 semillas) = −0.21, bajan 12 de 18, IC [−0.46, −0.12]. Diferencias de ±0.2 entre versiones son ruido de
semillas aunque parezcan "significativas".

**14c · Oráculo: corrección de 12d y variantes con 3 semillas** (todas frente a `lgb_future_clear_ses`, 3 semillas,
sin suavizar)

| Modelo (3 semillas + sesión) | CV | peor controlador | por controlador: media / mediana | suben / bajan | IC 90 % | escenarios 1 / 2 / 3 |
|---|---|---|---|---|---|---|
| oráculo (`lgb_future_clear_or_ses`) | **0.9236** | 0.763 | **+1.21** / +0.77 | 14 / 4 | [+0.40, +2.07] | +1.2 / +0.8 / +0.9 |
| **oráculo solo en coordinaciones** (`--oraculo-var coord`) | 0.9220 | **0.784** | +0.94 / +0.74 | 14 / 4 | [+0.36, +1.50] | +0.4 / +1.3 / +0.6 |
| oráculo + 63 hojas + `extra_trees` (`--extra-trees`) | 0.9188 | 0.787 | +0.86 / +0.28 | 9 / 8 | [+0.03, +1.74] | +0.7 / +0.9 / −0.3 |
| 10 semillas: oráculo frente a sin oráculo | 0.9215 frente a 0.9158 | 0.763 | +0.86 / +0.38 | 10 / 8 | [+0.01, +1.76] | +0.8 / +0.6 / +0.3 |

- **Corrección de 12d:** sin suavizar, el oráculo mejora **en los tres escenarios** (también en el 3). El "0.0 en el
  escenario 3" de 12d venía del suavizado y del ruido. La ganancia es más robusta de lo que se dijo allí.
- **Oráculo + 63 hojas + `extra_trees` no se suman** (0.9188 < 0.9236): corrigen los mismos casos.
- **Oráculo de coordinaciones frente a completo** (directo): −0.27 de media, IC [−0.89, +0.35], suben 6 / bajan 12.
  El completo gana algo en media; el de coordinaciones **protege al peor controlador** (0.784 frente a 0.763), porque
  no toca zone_conflict ni los conflictos, donde el oráculo restaba. Es un equilibrio entre ganancia y riesgo.
- **La alineación no explica lo del escenario 3:** allí es la mejor (desviación mediana de 0 s frente a 9 s y 2.5 s), y
  el acuerdo entre controladores es igual en los tres (53–58 % trabajan la misma tarea a ±180 s; 24–26 % en coordinaciones).

In [ ]:
# Comparaciones de esta sección (no reentrena nada; ~10 s cada una)
sys.path.insert(0, str(ROOT / "futuro" / "pruebas"))
from comparar import comparar
for c in ["lgb_future_clear_or_ses", "lgb_future_clear_orcoord_ses", "lgb_future_clear_or_l63et_ses"]:
    comparar("lgb_future_clear_ses", c)
comparar("lgb_future_clear_or_ses", "lgb_future_clear_orcoord_ses")

### 14d · Oráculo de coordinaciones con 10 semillas: nuevo mejor modelo (21/10, 01:15)

`python futuro/m_lgb_future.py --feats base,logic,past,future,clear --sesion --oraculo --oraculo-var coord --seeds 10 --name lgb_future_clear10_orcoord`

Todas las métricas de la regla de la cabecera, frente a `lgb_future_clear10_ses` (10 semillas, sin oráculo, sin suavizar):

| | CV | Δ | Δ mediana por sesión | sesiones suben / bajan | controladores suben / bajan | Δ por controlador (media / mediana) | IC 90 % | peor controlador | escenarios 1 / 2 / 3 |
|---|---|---|---|---|---|---|---|---|---|
| `lgb_future_clear10_ses` (referencia) | 0.9158 | — | — | — | — | — | — | 0.781 (001) | — |
| oráculo completo (`lgb_future_clear10_or_ses`) | 0.9215 | +0.57 | +0.00 | 18 / 17 | 10 / 8 | +0.86 / +0.38 | [+0.01, +1.76] | 0.763 (001) | 0.908 / 0.927 / 0.930 |
| **oráculo de coordinaciones (`lgb_future_clear10_orcoord_ses`)** | **0.9244** | **+0.86** | **+0.99** | **25 / 9** | **14 / 4** | **+1.13 / +1.06** | **[+0.47, +1.80]** | **0.790 (001)** | 0.905 / 0.936 / 0.933 |

- **Cumple la regla de adopción:** mejora la media **y** la mediana por sesión, sube en 25 sesiones y baja en 9, y el peor
  controlador también mejora.
- Con 10 semillas, el oráculo completo deja la mediana por sesión igual (18 / 17): era más frágil de lo que parecía con 3.
  Directo, coordinaciones frente a completo: +0.29, controladores 11 / 7.
- **Suavizado `prob` encima** (CV anidada): 0.9244 → **0.9251** (+0.07); parámetros (1.0, 2) en 4 de 5 folds; peor
  controlador 0.796; frente a `suav_lgb_future_clear10_ses`: sesiones 23 / 9, Δ mediana por sesión +0.57.
- Para comparar: `suav_f5_clear_l63_10_ses` (Juanma) frente a la misma referencia: +0.26, mediana por sesión +0.00, 16 / 17.

**Entregas finales recomendadas** (Kaggle se queda con la mejor de las dos seleccionadas):
1. **`futuro/submissions/sub_suav_lgb_future_clear10_orcoord_ses.csv`**: la mejor (CV anidada 0.925, peor controlador 0.796).
2. `futuro/submissions/sub_suav_lgb_future_clear10_ses.csv`: la segura, sin oráculo (CV 0.918, público 0.947).

### 14e · Kaggle: el oráculo de coordinaciones saca 0.934 en el público (21/10, 01:25)

| Envío | CV | público (020: sesiones 020sc2 y 020sc3) |
|---|---|---|
| sin oráculo (`sub_suav_lgb_future_clear10_ses`) | 0.918 | **0.947** |
| oráculo completo (`sub_suav_lgb_future_clear10_or_ses`) | 0.923 | 0.940 |
| oráculo de coordinaciones (`sub_suav_lgb_future_clear10_orcoord_ses`) | **0.925** | 0.934 |

**¿Qué probabilidad tiene −1.3 según la CV?** Δ por sesión (con oráculo de coordinaciones − sin oráculo, ambos 10
semillas + sesión + suavizado), en todas las parejas (sesión del escenario 2, sesión del escenario 3), como las del público:
media +0.98; **P(Δ medio ≤ −1.3) = 0.08**; P(Δ ≤ 0) = 0.21.

Δ por sesión en la CV: de −5.5 (002sc3) a +7.1 (004sc1). Hay controladores en los que el oráculo hace daño (002: −5.5
y −2.0; 018sc1: −3.7). El 020 ganaba +3.0 en su sesión de CV (020sc1).

**Lectura:** evidencia moderada de que en el 020 el oráculo no ayuda (dos envíos por debajo; el último, en el 8 % de
la cola). **No** indica un fallo en test (entonces caería muy por debajo de 0.92), y la CV completa sigue en +0.9.

**Decisión:** se refuerza la cobertura. Las 2 entregas finales son **una con oráculo y otra sin él** (Kaggle se queda
con la mejor):
1. `sub_suav_lgb_future_clear10_orcoord_ses.csv`: mejor CV con todos los criterios (14d).
2. `sub_suav_lgb_future_clear10_ses.csv`: sin oráculo, la mejor en el público.
No elegir dos versiones con oráculo.

### 14f · Diagnóstico del oráculo en el controlador 020 (21/10, 01:50; sin reentrenar)

**¿Hay un fallo técnico en las sesiones del 020?** No.
- **Alineación:** corrección 0 s y dispersión 0 s en 020sc2 y 020sc3. En train/val, la mediana es 0 s; el caso más raro es
  016sc1 (−103 s). En test, **023sc1 tiene −47 s** (dentro de lo visto en train).
- **Valores del oráculo en sus coordinaciones:** `or_near180` > 0 en el 21 % (020sc2) y el 17 % (020sc3), frente a cuartiles
  del 18–29 % en las sesiones etiquetadas; 023: 17–18 %. Normal.

**¿Qué hace distinto el oráculo en el 020?** (test, `sub_suav_lgb_future_clear10_orcoord_ses` frente a
`sub_suav_lgb_future_clear10_ses`)

| Sesión | snapshots con top-3 distinto | entry_coordination: salen / entran | exit_coordination: salen / entran | top-1 = coordinación (sin → con oráculo) |
|---|---|---|---|---|
| 020sc2 | 51 % | 133 / 53 | 77 / 53 | 9.1 % → 9.3 % |
| 020sc3 | 49 % | 118 / 25 | 60 / 39 | 11.1 % → 10.4 % |
| 023sc1 | 33 % | 19 / 12 | 47 / 30 | 3.3 % → 3.3 % |
| 023sc2 | 38 % | 49 / 8 | 41 / 22 | 5.2 % → 5.2 % |
| 023sc3 | 38 % | 53 / 0 | 52 / 62 | 3.4 % → 1.5 % |

- En el 020 el oráculo **cambia más** y, sobre todo, **saca entry_coordination del top-3**.
- **El 020 es muy de coordinaciones:** el 21 % de sus respuestas en 020sc1 (mediana 13 %, rango 7–26 %). El indicador sin
  etiquetas "top-1 del modelo sin oráculo = coordinación" correlaciona 0.80 con la fracción real por sesión, y le da 9–11 %
  en test frente al 3–5 % del 023.

**¿Castiga el oráculo a los controladores con muchas coordinaciones?** La CV **no** lo apoya: Spearman(fracción de
coordinaciones, ganancia del oráculo) = +0.18 por controlador (p = 0.47) y −0.10 por sesión. Por terciles de controladores,
los de muchas coordinaciones ganan **más** (+1.68, frente a +1.07 y +0.2).

**Conclusión:** no se puede atribuir el 0.934 del público a ese mecanismo. Lo más defendible es ruido más algo propio del
020. Para el 023 (pocas coordinaciones según el indicador), el oráculo cambia menos predicciones y pesa menos, para bien o
para mal. **Las entregas finales no cambian:** `sub_suav_lgb_future_clear10_orcoord_ses.csv` +
`sub_suav_lgb_future_clear10_ses.csv`.

### 14g · Selector de modelos por escenario con el oráculo de coordinaciones como candidato (21/10, 02:00; sin reentrenar)

**Ya probado por Juanma** (sección 15, `f6_ideas.py escen`): selección **anidada** por escenario (elige en 4 folds y mide
en el 5.º) entre A = `suav_lgb_future_clear10_ses`, L = `suav_f5_clear_l63_10_ses` y O = `suav_lgb_future_clear10_or_ses`
→ 0.9226, sesiones 13 / 13, frágil. Lo único nuevo aquí es añadir **C = `suav_lgb_future_clear10_orcoord_ses`** como
candidato (mismo procedimiento, sobre los mismos OOF suavizados).

| Candidatos | CV anidada | frente a C solo | peor controlador | sesiones frente a A (suben / bajan) | elegidos por escenario (folds 0–4) |
|---|---|---|---|---|---|
| A + L + O (Juanma) | 0.9226 | −0.33 | 0.769 | 13 / 13 | 1: OOOOO · 2: LLLOL · 3: OAAAO |
| A + L + C | 0.9229 | −0.30 | 0.784 | 20 / 9 | 1: CACCC · 2: CCCCL · 3: CCCCC |
| A + L + O + C | 0.9239 | −0.20 | 0.769 | 19 / 14 | 1: OOOOO · 2: CCCOL · 3: CCCCC |
| A + C | 0.9241 | −0.18 | 0.784 | 21 / 8 | 1: CACCC · 2: CCCCC · 3: CCCCC |
| C + L | 0.9247 | −0.12 | 0.796 | 22 / 10 | 1: CCCCC · 2: CCCCL · 3: CCCCC |
| **C solo** | **0.9259** (misma escala; 0.9251 en CV anidada) | — | **0.796** | 23 / 9 | — |

**Conclusión:** con C entre los candidatos, el selector lo elige casi siempre y en los tres escenarios. Elegir por
escenario solo añade decisiones que cambian entre folds y resta (−0.12 a −0.33). La ventaja de L en el escenario 2 y la
debilidad de O en el 3 no son sistemáticas. **Se queda C solo** (`orcoord`, 14d).

## 15 · Otros ámbitos: secuencia, controlador nuevo, etiquetas ruidosas, tramo y stacks con oráculo (Juanma, 21/10, 00:40–01:45)

Scripts: `futuro/pruebas/f6_ideas.py` (post-procesos sin reentrenar: `hmm`, `prior`, `escen`), `futuro/pruebas/f6_soft.py`
(1.er pase con etiquetas ruidosas), `futuro/pruebas/f7_tramo.py` (modelo por racha) y `f5_stack.py` (stacks). Registro en
`futuro/pruebas/resultados.jsonl`, hilos `f6`, `f7`, `f5_stack`. Referencia A = `suav_lgb_future_clear10_ses` (0.9181).

| Idea | Qué hace | Resultado | Veredicto |
|---|---|---|---|
| **HMM en la racha** (`hmm`) | cadena oculta: sigue con la tarea con prob. *s* o cambia; emisión p^τ; forward-backward; *s*, τ con CV anidada | sobre `clear10_ses` 0.9172; sobre `l63_10_ses` 0.9207 (suavizado: 0.9181 / 0.9202) | igual que el suavizado |
| **Adaptación al controlador** (`prior`, EM de Saerens) | reescala p(tarea) por (π_controlador / π_resto)^λ del tipo, estimado sin etiquetas | λ = 0 en los 5 folds | sin señal |
| **Etiquetas ruidosas** (`f6_soft.py`) | relevancia 2 = respuesta, 1 = respuesta a ±k snapshots (`grado1/3`); o quitar respuestas aisladas | `grado3` 0.9092 → 0.9151 con sesión; `grado1` 0.9133; `aislada` 0.9119 (control 0.9071 → 0.9142) | ruido |
| **Modelo por escenario** (`escen`) | por escenario, el mejor de A / l63 / oráculo en los folds de entrenamiento | 0.9226, mediana por sesión 0, sesiones 13 / 13 | frágil |
| **Modelo por racha** (`f7_tramo.py`) | LambdaRank sobre (racha, tarea) con agregados del 1.er pase y de `future`/`clear`; score = lsm + w · racha | solo racha 0.898; combinado 0.9064 (1.er pase 0.9086); + sesión 0.9147 (0.9158) | no aporta |
| **Stack con oráculo** (`stackf_or`) | `clear10_or_ses` 0.47, `l63_10_ses` 0.12, `linear_ses` 0.18, `f4_ses` 0.24; + suavizado | **0.9256** (anidada), peor controlador 0.781, sesiones 23 / 7, controladores 14 / 4 | empata con `orcoord` (14d) |
| Stack con oráculo de coordinaciones (`stackf_orc`, `stackf_orc2`) | lo mismo con `clear10_orcoord_ses` | 0.9204 / 0.9214 (`orcoord` solo: 0.9244) | **peor que el modelo solo** |

**Análisis de errores** (`suav_f5_clear_l63_10_ses`): la correcta está en top-3 el 92.1 %, en 4–5 el 2.9 %, en 6–10 el
3.4 %, más allá el 1.6 %. La mitad de los fallos son coordinaciones (exit 28.5 %, entry 22.8 %) y en esos casos el top-1
suele ser otra coordinación. Las rachas se aciertan o se fallan enteras (92.6 % todo bien, 3.8 % todo mal, 3.5 % mixtas):
por eso ningún post-proceso en la racha (suavizado, HMM, modelo por racha) arregla lo que queda.

**Conclusiones:**
- Con la información del snapshot, la sesión y la racha, los post-procesos están saturados (todos ±0.002).
- La única fuente nueva que suma es el oráculo (otros controladores). Stack con oráculo completo y `orcoord` solo empatan
  (`comparar.py`: IC 90 % [+0.27, +1.7] en los dos). Se prefiere **`orcoord`**: mejor peor controlador (0.796 frente a
  0.781) y un solo modelo; los pesos del stack no son estables (con `orcoord` dentro, el stack queda por debajo).


## 16 · Atacar las coordinaciones desde el modelo: peso, especialista y features relativas (21/10, 02:05–02:30)

Base: `lgb_future_clear_orcoord` (3 semillas; 1.er pase 0.9101, + sesión **0.9220**, peor controlador 0.784).
Todo con 3 semillas y medido contra esa base. Las ideas 4 (suavizado adaptativo) y 5 (modelo por racha) no se repiten:
equivalen al HMM y a `f7_tramo.py` de la sección 15.

| Prueba | Qué cambia | 1.er pase | + sesión | Δ | Δ mediana por sesión | sesiones suben / bajan | controladores suben / bajan | peor controlador |
|---|---|---|---|---|---|---|---|---|
| peso ×1.5 (`--peso-coord 1.5`) | los snapshots cuya respuesta es una coordinación pesan 1.5 en el ranking | 0.9101 | 0.9211 | −0.09 | +0.00 | 15 / 14 | 6 / 9 | 0.778 |
| peso ×2.0 (`--peso-coord 2.0`) | ídem, 2.0 | 0.9093 | 0.9233 | +0.13 | +0.20 | 20 / 10 | 10 / 7 | 0.775 |
| bloque `rel` (`futuro/feat_rel.py`) | diferencia con el mejor competidor del mismo tipo / snapshot (siguiente instrucción, desaparición, `sx_task`), rango dentro del tipo, nº de tareas del avión que encajan con la siguiente instrucción, nº de aviones con instrucción en ≤ 30 s | 0.9094 | 0.9236 | +0.17 | +0.00 | 17 / 14 | 9 / 7 | 0.775 |
| especialista (`futuro/pruebas/especialista.py`) | ranker solo con candidatos de coordinación; base + β · especialista en esas filas, β en CV anidada | — | 0.9211 (anidada) | −0.09 | — | — | — | — |

- **Peso:** ×1.5 y ×2.0 se mueven en sentidos opuestos (−0.09 / +0.13), y la media por controlador no cambia: ruido de semillas.
- **`rel`:** la señal existe (la correcta tiene la siguiente instrucción 34 s antes que su mejor competidor del mismo
  tipo; el resto, 167 s después), pero el modelo ya la sacaba de los rangos que había. +0.17 con la mediana por sesión
  igual y el peor controlador peor: no se adopta.
- **Especialista:** la CV anidada elige β = 0 en 4 de 5 folds; con β > 0, cuanto más pesa, peor (0.25 → 0.9212; 0.5 →
  0.9113; 1 → 0.8908; 2 → 0.8437). Entrenado solo con coordinaciones, no sabe cuándo la respuesta NO es una coordinación.

**Conclusión:** ninguna de las tres cumple la regla de adopción. El modelo no saca más de las coordinaciones con la
información disponible; lo único que ha sumado en ellas es el oráculo. **Se queda `orcoord` (14d).**

*(Nota de código: desde aquí `--ref` es el nombre del 1.er pase de referencia; el 1.er pase se compara con `<ref>` y el
segundo paso con `<ref>_ses`. Antes se comparaba el 1.er pase con `<ref>_ses`; esas líneas del registro no son válidas.)*

## 17 · Oráculo de coordinaciones ponderado por estilo y auditoría de ambigüedad (21/10)

Base: `lgb_future_clear_orcoord` (3 semillas; + sesión **0.9220**, peor controlador 0.784). Script de los diagnósticos:
`futuro/pruebas/auditoria_coord.py` (sin reentrenar). El modelo de eventos temporales (hazard) **no se probó**: el segundo
paso de sesión ya modela que cada tarea se trabaja una vez y el suavizado modela las rachas.

**a) ¿Los controladores de estilo parecido coinciden más?** Perfil de estilo por sesión **sin etiquetas**
(`feat_oracle.estilo`, z-scores): cuándo asume y transfiere respecto al borde del sector, ritmo de instrucciones y mezcla
de tipos (nivel, lateral, rumbo, asunción forzada). En 424 parejas de sesiones del mismo escenario y distinto controlador,
Spearman(distancia de estilo, acuerdo de respuestas a ±180 s) = **−0.20** (p = 2e-5). El acuerdo es 0.575 con el tercio de
referencias más parecidas, frente a 0.529 con el menos parecido. La señal es real, pero pequeña.
- **023 (privado) es atípico:** 023sc2 es más atípica que cualquier sesión etiquetada (percentil 1.00; `frac_forzado`
  z = 3.51, `tts_al_asumir` z = 2.48) y 023sc1 está en el percentil 0.92 (`tts_al_asumir` z = 3.55). 020 es típico
  (percentiles 0.14 y 0.19).

**b) Oráculo ponderado** (`--oraculo-var coord --oraculo-temp T`): cada sesión fuente pesa exp(−distancia de estilo / T)
en vez de pesar todas igual; T = ∞ es el uniforme.

| T | 1.er pase | + sesión | Δ | Δ mediana por sesión | sesiones suben / bajan | controladores suben / bajan | peor controlador | P(ganancia media > 0), bootstrap |
|---|---|---|---|---|---|---|---|---|
| ∞ (uniforme) | 0.9101 | 0.9220 | — | — | — | — | 0.784 (001) | — |
| 2 | 0.9120 | 0.9230 | +0.10 | +0.00 | 15 / 15 | 9 / 6 | 0.790 (001) | 0.52 |
| 1 | 0.9113 | 0.9215 | −0.05 | +0.00 | 14 / 15 | 8 / 7 | 0.743 (001) | 0.14 |
| 0.5 | 0.9094 | 0.9218 | −0.02 | +0.00 | 16 / 16 | 5 / 11 | 0.781 (001) | 0.21 |

- **CV anidada** (elige T en 4 folds y mide en el 5.º, entre ∞, 2, 1 y 0.5): T = 2 en los 5 folds → **0.9230**. Son +0.10
  sobre el uniforme, dentro del ruido de semillas (±0.2, sección 14b), con la mediana por sesión sin cambios y la ganancia
  media por controlador en +0.00.
- **¿Ayuda a las sesiones atípicas, que es lo que importaría para 023?** No. Spearman(atipicidad, Δ por sesión) = +0.12
  (p = 0.48) con T = 2, y −0.01 / −0.06 con T = 1 / 0.5. En las 8 sesiones más atípicas, Δ = +0.36 con T = 2 (resto +0.04),
  pero con T más bajo, que pondera más, el efecto desaparece en vez de crecer.

**c) Auditoría de ambigüedad de las coordinaciones.** Snapshots cuya respuesta es una coordinación, mirando solo las
candidatas de su mismo tipo. Se compara el modelo (`lgb_future_clear_orcoord`) con un kNN (25 vecinos, por folds) sobre las
mismas features.

| | snapshots | top-1 al azar | top-1 modelo | top-1 kNN | AUC por parejas modelo | AUC kNN | vecino más parecido con etiqueta contraria (10 % más cercanos) |
|---|---|---|---|---|---|---|---|
| entry_coordination | 1068 | 0.105 | **0.727** | 0.546 | **0.950** | 0.863 | 20 % (n = 10) |
| exit_coordination | 736 | 0.225 | **0.701** | 0.433 | **0.851** | 0.691 | 42 % (n = 52) |

- El modelo ya separa mucho mejor que los vecinos: estos no traen señal que el modelo no use.
- En exit_coordination, el 42 % de los contextos casi idénticos de otro controlador tienen la etiqueta contraria. Hay
  un **techo de información**: qué coordinación concreta se hace depende del controlador, no de lo observable.

**Conclusión:** no se adopta el oráculo ponderado; se queda `orcoord` uniforme. La similitud de estilo existe pero es
demasiado débil para mejorar el promedio, y 023 está fuera del rango de estilos vistos, así que ponderar por parecido
tampoco tiene referencias cercanas en las que apoyarse. La auditoría explica por qué las secciones 11 y 16 no lograron
subir las coordinaciones: la ambigüedad que queda está en las etiquetas, no en el modelo. No se gastan 10 semillas en T = 2.

In [ ]:
# Reproducir (terminal, desde la raíz):
#   for T in 2 1 0.5; do python futuro/m_lgb_future.py --feats base,logic,past,future,clear --sesion --oraculo \
#       --oraculo-var coord --oraculo-temp $T --threads 10 --ref lgb_future_clear_orcoord --name lgb_future_clear_orcoord_t${T/./}; done
#   python futuro/pruebas/auditoria_coord.py          # a) estilo, b) atipicidad frente a Δ, c) ambigüedad (~3 min)
# Comparaciones (no reentrena nada):
from comparar import comparar
for t in ("t2", "t1", "t05"):
    comparar("lgb_future_clear_orcoord_ses", f"lgb_future_clear_orcoord_{t}_ses")

## 18 · Información nueva: geometría del tráfico y adaptación al controlador por covariate shift (21/10)

Referencia: `lgb_future_clear` **sin oráculo** (3 semillas, mismos folds y semillas; + sesión **0.9142**, peor controlador
0.781). Las columnas `entry` y `exit` dan el acierto top-3 en los snapshots cuya respuesta es entry_coordination o
exit_coordination.

**a) Geometría** (`futuro/feat_geo.py`, bloques `geoA`, `geoB`, `geoC`). Los datos **no traen posiciones** (ni lat/lon ni
rumbo), así que no hay CPA/TCPA entre parejas de aviones. Con lo que hay por avión:
- **A · borde del sector:** velocidad real de acercamiento al borde en los últimos ~30 s, tiempo hasta cruzar a esa
  velocidad y orden de entrada y salida entre los aviones del snapshot.
- **B · tráfico que cruza el mismo borde a la vez:** nº de otros aviones que entran o salen a ±60/120 s, cuántos a ≤ 10/20 FL
  y mínima diferencia de tiempo y de nivel con ellos.
- **C · niveles:** velocidad vertical, FL que faltan hasta el de salida y ritmo necesario, nivel autorizado frente al de
  salida, y segundos desde el último cambio del nivel de salida y hasta el siguiente.

| Bloques | + sesión | Δ | Δ mediana por sesión | sesiones suben / bajan | controladores suben / bajan | peor controlador | entry | exit |
|---|---|---|---|---|---|---|---|---|
| referencia | 0.9142 | — | — | — | — | 0.781 (001) | 0.754 | 0.611 |
| A | 0.9139 | −0.02 | +0.00 | 17 / 16 | 9 / 8 | 0.766 (001) | 0.744 | 0.628 |
| B | 0.9103 | −0.39 | +0.00 | 14 / 18 | 5 / 11 | 0.775 (001) | 0.740 | 0.611 |
| C | 0.9132 | −0.10 | +0.00 | 16 / 15 | 8 / 8 | 0.784 (001) | 0.739 | 0.630 |
| A + B | 0.9120 | −0.21 | +0.00 | 16 / 16 | 5 / 12 | 0.775 (001) | 0.751 | 0.605 |
| A + B + C | 0.9126 | −0.15 | −0.24 | 13 / 19 | 8 / 8 | 0.766 (001) | 0.741 | 0.626 |

- Ningún bloque mejora. A y C suben exit_coordination (+1.7 / +1.9) pero bajan entry_coordination en una cantidad parecida.
- B, la interacción física, es el que más empeora: con ±120 s y ±10 FL marca sobre todo la densidad de tráfico, que ya
  está en `seq_len`, `l_n_*` y `logic_n_window`.

**b) Covariate shift** (`futuro/pruebas/covshift.py`). Para cada controlador de validación, un clasificador de dominio
(solo variables de entrada) estima qué snapshots de entrenamiento se parecen a ese controlador. El ranker se reentrena con
pesos w^α, con w recortado a [0.1, 10]; α = 0 es la referencia. Es nuevo frente a la sección 15: aquello reescalaba
P(tipo) y esto repondera P(X).

| α | 1.er pase | + sesión | Δ | Δ mediana por sesión | sesiones suben / bajan | controladores suben / bajan | peor controlador |
|---|---|---|---|---|---|---|---|
| 0 (referencia) | 0.9071 | 0.9142 | — | — | — | — | 0.781 |
| 0.5 | 0.9053 | 0.9110 | −0.32 | −0.25 | 8 / 21 | 6 / 11 | 0.778 |
| 1 | 0.9039 | 0.9131 | −0.11 | +0.20 | 19 / 15 | 9 / 9 | 0.766 |

- **CV anidada:** elige α = 0 en 4 de los 5 folds → 0.9127 (−0.15).
- **Por qué no puede funcionar:** con las filas del controlador dentro del ajuste, el clasificador lo separa «perfectamente»
  (AUC 0.999–1.000) y los pesos degeneran (tamaño efectivo 17–23 %). Pero medido **fuera de muestra**, en la otra mitad
  temporal de las sesiones del controlador, el AUC es 0.55 en el 020 y 0.69 en el 001 (0.48–0.56 con un clasificador más
  débil). Lo que distingue a un controlador son rasgos de la sesión (`blk_len`, `seq_len`, `a_n300`), no un dominio
  estable de P(X).
- Igual que la auditoría de la sección 17: entre controladores casi no cambia lo observable; cambia la decisión ante el
  mismo contexto, y eso no se corrige reponderando.

**c) Beam search sobre los OOF: no se prueba.** Una búsqueda en haz da el camino más probable de la sesión, pero el top-3
se evalúa snapshot a snapshot y necesita la probabilidad de cada tarea en cada snapshot. Eso lo calcula exactamente
forward-backward, que es el HMM de la sección 15 (sin mejora); el segundo paso de sesión ya penaliza las repeticiones.

**Conclusión:** no se adopta nada. La geometría disponible ya está en las features (los datos no traen posiciones) y no
hay un cambio de distribución estable entre controladores que aprovechar. Las entregas no cambian.

In [ ]:
# Reproducir (terminal, desde la raíz; ~6 min cada una, se pueden lanzar a la vez con --threads 6):
#   for v in A B C AB ABC; do g=$(echo $v | sed 's/./,geo&/g'); python futuro/m_lgb_future.py \
#       --feats base,logic,past,future,clear$g --sesion --threads 6 --name lgb_future_clear_geo$v; done
#   python futuro/pruebas/covshift.py 0.5 1 --threads 15     # ~17 min
from comparar import comparar
for c in ["lgb_future_clear_geoA_ses", "lgb_future_clear_geoB_ses", "lgb_future_clear_geoC_ses",
          "lgb_future_clear_geoAB_ses", "lgb_future_clear_geoABC_ses", "covshift_a0.5_ses", "covshift_a1_ses"]:
    comparar("lgb_future_clear_ses", c)

## Historial completo

In [ ]:
historial()